In [16]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.linear_model import LogisticRegression,SGDClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import MinMaxScaler

current_dir = Path.cwd()
project_root = current_dir.parents[2]

full_set_path_HY3 = project_root / 'SCRIPTS/PYTHON/MODEL ANALYSIS/MODEL_PERFORMANCE_RESULTS/CLASSIFICATION_UPDRS_DATA/HY3/FULL_SET/'
full_set_path_HY4 = project_root / 'SCRIPTS/PYTHON/MODEL ANALYSIS/MODEL_PERFORMANCE_RESULTS/CLASSIFICATION_UPDRS_DATA/HY4/FULL_SET/'
full_set_path_MCID = project_root / 'SCRIPTS/PYTHON/MODEL ANALYSIS/MODEL_PERFORMANCE_RESULTS/CLASSIFICATION_UPDRS_DATA/MCID/FULL_SET/'

feature_selection_path_HY3 = project_root / 'SCRIPTS/PYTHON/MODEL ANALYSIS/MODEL_PERFORMANCE_RESULTS/CLASSIFICATION_UPDRS_DATA/HY3/FEATURE_SELECTION/'
feature_selection_path_HY4 = project_root / 'SCRIPTS/PYTHON/MODEL ANALYSIS/MODEL_PERFORMANCE_RESULTS/CLASSIFICATION_UPDRS_DATA/HY4/FEATURE_SELECTION/'
feature_selection_path_MCID = project_root / 'SCRIPTS/PYTHON/MODEL ANALYSIS/MODEL_PERFORMANCE_RESULTS/CLASSIFICATION_UPDRS_DATA/MCID/FEATURE_SELECTION/'


from lightgbm import LGBMClassifier
from sklearn.linear_model import SGDClassifier

classification_models = {

    "decision_tree": DecisionTreeClassifier(random_state=42),

    "random_forest": RandomForestClassifier(
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    ),

    "extra_trees": ExtraTreesClassifier(
        random_state=42,
        n_jobs=-1
    ),

    "xgboost": XGBClassifier(
        tree_method="hist",
        eval_metric="logloss",
        n_jobs=-1,
        random_state=42
    ),


    "adaboost": AdaBoostClassifier(
        algorithm="SAMME", 
        random_state=42
    ),

    "svm": SVC(
        kernel="rbf",
        probability=True,  
        random_state=42
    ),

    "logistic_regression": LogisticRegression(
        random_state=42,
        n_jobs=-1,
        max_iter=10000,
        class_weight="balanced"
    ),

    "knn": KNeighborsClassifier(
        n_jobs=-1
    ),

    "gaussian_nb": GaussianNB()
}

In [17]:
import json

with open(project_root/"SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS_Domain_data.json", "r") as archivo:
    datos = json.load(archivo)

print('Subject Domain:', len(datos['SC_data']))
SC_cols=datos['SC_data']
print('Non Motor Domain:', len(datos['UPDRS_NO_MOTOR']))
NM_cols=datos['UPDRS_NO_MOTOR']
print('Impact Motor Domain:', len(datos['UPDRS_IMPACTO_MOTOR']))
IM_cols=datos['UPDRS_IMPACTO_MOTOR']
print('Motor Domain:', len(datos['UPDRS_MOTOR']))
M_cols=datos['UPDRS_MOTOR']
print('Total UPDRS 3 Features:', len(datos['UPDRS_MOTOR']) + len(datos['UPDRS_IMPACTO_MOTOR']) + len(datos['UPDRS_NO_MOTOR']))

Subject Domain: 6
Non Motor Domain: 65
Impact Motor Domain: 65
Motor Domain: 165
Total UPDRS 3 Features: 295


# ANALYISIS DEFAULT MODELS


In [18]:
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
import numpy as np
import pandas as pd


def evaluate_models_10x10_oof_and_test(
    X_df: pd.DataFrame,
    y_df: pd.DataFrame,
    models: dict,
    scaler = StandardScaler(),
    outer_splits: int = 10,
    inner_splits: int = 10,
    test_size: float = 0.3,
    random_state: int = 42,
    decimals: int = 4,
):
    y = y_df.iloc[:, 0].to_numpy()
    X = X_df.to_numpy()
    classes = np.unique(y)

    def build_pipeline(estimator):
        return Pipeline([
            ("scaler", scaler),
            ("model", clone(estimator)),
        ])

    def compute_metrics(y_true, y_pred, y_proba):
        return {
            "Accuracy": accuracy_score(y_true, y_pred),
            "Precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
            "Recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
            "F1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
            "AUC_macro": roc_auc_score(y_true, y_proba, multi_class="ovr", average="macro"),
            "Precision_weighted": precision_score(y_true, y_pred, average="weighted", zero_division=0),
            "Recall_weighted": recall_score(y_true, y_pred, average="weighted", zero_division=0),
            "F1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
            "AUC_weighted": roc_auc_score(y_true, y_proba, multi_class="ovr", average="weighted"),
        }

    def summarize(metrics_list, suffix):
        df = pd.DataFrame(metrics_list)
        mean = df.mean()
        std = df.std(ddof=1)
        return {
            f"{col}_{suffix}": f"{mean[col]:.{decimals}f} ± {std[col]:.{decimals}f}"
            for col in df.columns
        }


    outer = StratifiedShuffleSplit(
        n_splits=outer_splits,
        test_size=test_size,
        random_state=random_state
    )

    test_summary_rows = []
    cv_summary_rows = []

    for model_name, estimator in models.items():
        print(f"Evaluating {model_name}...")

        test_metrics_all = []
        cv_metrics_all = []

        for train_idx, test_idx in outer.split(X, y):
            X_train, y_train = X[train_idx], y[train_idx]
            X_test, y_test = X[test_idx], y[test_idx]

            inner = StratifiedKFold(
                n_splits=inner_splits,
                shuffle=True,
                random_state=random_state
            )

            oof_pred = np.zeros(len(y_train), dtype=y_train.dtype)
            oof_proba = np.zeros((len(y_train), len(classes)))

            for tr_idx, val_idx in inner.split(X_train, y_train):
                X_tr, y_tr = X_train[tr_idx], y_train[tr_idx]
                X_val = X_train[val_idx]

                model = build_pipeline(estimator)
                model.fit(X_tr, y_tr)

                oof_pred[val_idx] = model.predict(X_val)

                fold_proba = model.predict_proba(X_val)
                fold_classes = model.named_steps["model"].classes_

                aligned_proba = np.zeros((len(val_idx), len(classes)))
                for j, cls in enumerate(fold_classes):
                    aligned_proba[:, np.where(classes == cls)[0][0]] = fold_proba[:, j]

                oof_proba[val_idx] = aligned_proba

            cv_metrics_all.append(compute_metrics(y_train, oof_pred, oof_proba))

            model_full = build_pipeline(estimator)
            model_full.fit(X_train, y_train)

            test_pred = model_full.predict(X_test)
            test_proba_raw = model_full.predict_proba(X_test)
            test_classes = model_full.named_steps["model"].classes_

            test_proba = np.zeros((len(y_test), len(classes)))
            for j, cls in enumerate(test_classes):
                test_proba[:, np.where(classes == cls)[0][0]] = test_proba_raw[:, j]

            test_metrics_all.append(compute_metrics(y_test, test_pred, test_proba))

        test_summary_rows.append(
            pd.Series(summarize(test_metrics_all, "Testing"), name=model_name)
        )
        cv_summary_rows.append(
            pd.Series(summarize(cv_metrics_all, "CV"), name=model_name)
        )

    df_test_summary = pd.DataFrame(test_summary_rows)[[
        "Accuracy_Testing",
        "Precision_macro_Testing", "Recall_macro_Testing", "F1_macro_Testing", "AUC_macro_Testing",
        "Precision_weighted_Testing", "Recall_weighted_Testing", "F1_weighted_Testing", "AUC_weighted_Testing"
    ]]

    df_cv_summary = pd.DataFrame(cv_summary_rows)[[
        "Accuracy_CV",
        "Precision_macro_CV", "Recall_macro_CV", "F1_macro_CV", "AUC_macro_CV",
        "Precision_weighted_CV", "Recall_weighted_CV", "F1_weighted_CV", "AUC_weighted_CV"
    ]]

    df_final_summary = pd.concat([df_test_summary, df_cv_summary], axis=1)
    return df_final_summary

## HY3


### FULL DOMAIN


In [19]:
domain_cols = SC_cols + NM_cols + IM_cols + M_cols
X_HY3_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_HY3_full.csv", index_col=0)[domain_cols]
y_HY3_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY3_full.csv", index_col=0)
print("X_HY3_data shape:", X_HY3_data.shape)
print("y_HY3_data shape:", y_HY3_data.shape)
X_HY3_data.head()

X_HY3_data shape: (909, 301)
y_HY3_data shape: (909, 1)


,ENRLLRRK2,ENRLGBA,SEX,RAWHITE,EDUCYRS,AGE_AT_VISIT,NP1COG_mean,NP1COG_min,NP1COG_max,NP1COG_var,...,NP3RTALJ_mean,NP3RTALJ_min,NP3RTALJ_max,NP3RTALJ_var,NP3RTALJ_std,NP3RTCON_mean,NP3RTCON_min,NP3RTCON_max,NP3RTCON_var,NP3RTCON_std
PATNO,,,,,,,,,,,,,,,,,,,,,
3003,0,0,0.0,1.0,16.0,59.7,0.000000,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.666667,0.0,1.0,0.333333,0.577350
3018,0,0,0.0,1.0,16.0,63.6,0.333333,0.0,1.0,0.333333,...,0.0,0.0,0.0,0.0,0.0,2.666667,1.0,4.0,2.333333,1.527525
3020,0,0,0.0,1.0,15.0,77.0,1.000000,1.0,1.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,2.666667,2.0,4.0,1.333333,1.154701
3028,0,0,1.0,1.0,22.0,78.8,0.666667,0.0,1.0,0.333333,...,0.0,0.0,0.0,0.0,0.0,1.333333,0.0,3.0,2.333333,1.527525
3051,0,0,1.0,1.0,18.0,74.7,0.000000,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,1.333333,1.0,2.0,0.333333,0.577350


In [20]:
df_HY3_full = evaluate_models_10x10_oof_and_test(
    X_df=X_HY3_data, 
    y_df=y_HY3_data, 
    models=classification_models, 
    random_state=42
)

df_HY3_full.to_csv(full_set_path_HY3 / "HY3_full_StandardScaler.csv")

df_HY3_full_min_max = evaluate_models_10x10_oof_and_test(
    X_df=X_HY3_data, 
    y_df=y_HY3_data, 
    models=classification_models,
    scaler = MinMaxScaler(), 
    random_state=42
)
df_HY3_full_min_max.to_csv(full_set_path_HY3 / "HY3_full_MinMaxScaler.csv")


Evaluating decision_tree...
Evaluating random_forest...
Evaluating extra_trees...
Evaluating xgboost...
Evaluating adaboost...
Evaluating svm...
Evaluating logistic_regression...
Evaluating knn...
Evaluating gaussian_nb...
Evaluating decision_tree...
Evaluating random_forest...
Evaluating extra_trees...
Evaluating xgboost...
Evaluating adaboost...
Evaluating svm...
Evaluating logistic_regression...
Evaluating knn...
Evaluating gaussian_nb...


In [21]:
df_HY3_full.head(10)

,Accuracy_Testing,Precision_macro_Testing,Recall_macro_Testing,F1_macro_Testing,AUC_macro_Testing,Precision_weighted_Testing,Recall_weighted_Testing,F1_weighted_Testing,AUC_weighted_Testing,Accuracy_CV,Precision_macro_CV,Recall_macro_CV,F1_macro_CV,AUC_macro_CV,Precision_weighted_CV,Recall_weighted_CV,F1_weighted_CV,AUC_weighted_CV
decision_tree,0.8322 ± 0.0136,0.6116 ± 0.0366,0.6202 ± 0.0397,0.6133 ± 0.0351,0.7588 ± 0.0218,0.8337 ± 0.0141,0.8322 ± 0.0136,0.8319 ± 0.0123,0.8459 ± 0.0128,0.8283 ± 0.0130,0.6438 ± 0.0180,0.6526 ± 0.0251,0.6469 ± 0.0201,0.7737 ± 0.0132,0.8308 ± 0.0114,0.8283 ± 0.0130,0.8293 ± 0.0122,0.8422 ± 0.0117
random_forest,0.9007 ± 0.0116,0.6340 ± 0.1056,0.6223 ± 0.0162,0.6169 ± 0.0272,0.9434 ± 0.0143,0.8813 ± 0.0138,0.9007 ± 0.0116,0.8894 ± 0.0117,0.9588 ± 0.0065,0.8994 ± 0.0052,0.6499 ± 0.1103,0.6213 ± 0.0061,0.6154 ± 0.0126,0.9551 ± 0.0070,0.8784 ± 0.0075,0.8994 ± 0.0052,0.8870 ± 0.0047,0.9617 ± 0.0029
extra_trees,0.9004 ± 0.0168,0.7344 ± 0.1772,0.6354 ± 0.0311,0.6416 ± 0.0487,0.9531 ± 0.0122,0.8894 ± 0.0251,0.9004 ± 0.0168,0.8909 ± 0.0175,0.9623 ± 0.0064,0.9005 ± 0.0058,0.8261 ± 0.0646,0.6517 ± 0.0115,0.6686 ± 0.0197,0.9596 ± 0.0057,0.8952 ± 0.0075,0.9005 ± 0.0058,0.8917 ± 0.0051,0.9630 ± 0.0028
xgboost,0.8971 ± 0.0104,0.7599 ± 0.0721,0.6877 ± 0.0419,0.7072 ± 0.0511,0.9251 ± 0.0245,0.8919 ± 0.0133,0.8971 ± 0.0104,0.8923 ± 0.0114,0.9569 ± 0.0037,0.8942 ± 0.0075,0.7837 ± 0.0715,0.6896 ± 0.0207,0.7141 ± 0.0296,0.9368 ± 0.0117,0.8894 ± 0.0095,0.8942 ± 0.0075,0.8896 ± 0.0074,0.9552 ± 0.0041
adaboost,0.8766 ± 0.0235,0.7165 ± 0.0747,0.6999 ± 0.0512,0.7015 ± 0.0535,0.9139 ± 0.0130,0.8788 ± 0.0184,0.8766 ± 0.0235,0.8765 ± 0.0209,0.9222 ± 0.0126,0.8780 ± 0.0065,0.7510 ± 0.0447,0.7210 ± 0.0370,0.7308 ± 0.0356,0.9182 ± 0.0137,0.8768 ± 0.0071,0.8780 ± 0.0065,0.8769 ± 0.0068,0.9232 ± 0.0095
svm,0.8993 ± 0.0111,0.5995 ± 0.0074,0.6169 ± 0.0075,0.6076 ± 0.0075,0.9467 ± 0.0097,0.8772 ± 0.0106,0.8993 ± 0.0111,0.8874 ± 0.0110,0.9549 ± 0.0074,0.8972 ± 0.0081,0.5981 ± 0.0054,0.6165 ± 0.0055,0.6071 ± 0.0055,0.9504 ± 0.0052,0.8719 ± 0.0078,0.8972 ± 0.0081,0.8842 ± 0.0079,0.9568 ± 0.0035
logistic_regression,0.8513 ± 0.0260,0.6512 ± 0.0362,0.6661 ± 0.0460,0.6524 ± 0.0358,0.8909 ± 0.0297,0.8594 ± 0.0193,0.8513 ± 0.0260,0.8533 ± 0.0227,0.9162 ± 0.0126,0.8527 ± 0.0118,0.6808 ± 0.0347,0.6926 ± 0.0298,0.6857 ± 0.0324,0.8971 ± 0.0184,0.8559 ± 0.0108,0.8527 ± 0.0118,0.8540 ± 0.0112,0.9197 ± 0.0094
knn,0.8560 ± 0.0153,0.6780 ± 0.1396,0.6091 ± 0.0226,0.6092 ± 0.0382,0.8557 ± 0.0344,0.8576 ± 0.0146,0.8560 ± 0.0153,0.8456 ± 0.0150,0.9228 ± 0.0089,0.8572 ± 0.0067,0.7474 ± 0.1311,0.6159 ± 0.0223,0.6201 ± 0.0369,0.8646 ± 0.0077,0.8593 ± 0.0125,0.8572 ± 0.0067,0.8462 ± 0.0073,0.9213 ± 0.0056
gaussian_nb,0.7883 ± 0.0268,0.6479 ± 0.0137,0.7815 ± 0.0344,0.6337 ± 0.0225,0.9021 ± 0.0274,0.8881 ± 0.0159,0.7883 ± 0.0268,0.8177 ± 0.0243,0.9271 ± 0.0113,0.7953 ± 0.0275,0.6528 ± 0.0146,0.7824 ± 0.0400,0.6474 ± 0.0293,0.8855 ± 0.0231,0.8829 ± 0.0075,0.7953 ± 0.0275,0.8237 ± 0.0210,0.9225 ± 0.0068


In [22]:
df_HY3_full_min_max.head(10)

,Accuracy_Testing,Precision_macro_Testing,Recall_macro_Testing,F1_macro_Testing,AUC_macro_Testing,Precision_weighted_Testing,Recall_weighted_Testing,F1_weighted_Testing,AUC_weighted_Testing,Accuracy_CV,Precision_macro_CV,Recall_macro_CV,F1_macro_CV,AUC_macro_CV,Precision_weighted_CV,Recall_weighted_CV,F1_weighted_CV,AUC_weighted_CV
decision_tree,0.8319 ± 0.0147,0.6128 ± 0.0389,0.6200 ± 0.0396,0.6139 ± 0.0364,0.7584 ± 0.0217,0.8331 ± 0.0143,0.8319 ± 0.0147,0.8313 ± 0.0130,0.8453 ± 0.0131,0.8277 ± 0.0130,0.6411 ± 0.0201,0.6486 ± 0.0270,0.6436 ± 0.0229,0.7714 ± 0.0142,0.8297 ± 0.0116,0.8277 ± 0.0130,0.8284 ± 0.0122,0.8414 ± 0.0118
random_forest,0.9004 ± 0.0109,0.6336 ± 0.1057,0.6220 ± 0.0161,0.6165 ± 0.0271,0.9414 ± 0.0164,0.8807 ± 0.0131,0.9004 ± 0.0109,0.8889 ± 0.0109,0.9587 ± 0.0068,0.8997 ± 0.0050,0.6667 ± 0.1151,0.6233 ± 0.0079,0.6189 ± 0.0154,0.9553 ± 0.0069,0.8801 ± 0.0088,0.8997 ± 0.0050,0.8875 ± 0.0047,0.9616 ± 0.0028
extra_trees,0.9004 ± 0.0168,0.7344 ± 0.1772,0.6354 ± 0.0311,0.6416 ± 0.0487,0.9531 ± 0.0122,0.8894 ± 0.0251,0.9004 ± 0.0168,0.8909 ± 0.0175,0.9623 ± 0.0064,0.9005 ± 0.0058,0.8261 ± 0.0646,0.6517 ± 0.0115,0.6686 ± 0.0197,0.9596 ± 0.0057,0.8952 ± 0.0075,0.9005 ± 0.0058,0.8917 ± 0.0051,0.9630 ± 0.0028
xgboost,0.8971 ± 0.0104,0.7599 ± 0.0721,0.6877 ± 0.0419,0.7072 ± 0.0511,0.9251 ± 0.0245,0.8919 ± 0.0133,0.8971 ± 0.0104,0.8923 ± 0.0114,0.9568 ± 0.0037,0.8942 ± 0.0075,0.7837 ± 0.0715,0.6896 ± 0.0207,0.7141 ± 0.0296,0.9369 ± 0.0117,0.8894 ± 0.0095,0.8942 ± 0.0075,0.8896 ± 0.0074,0.9553 ± 0.0041
adaboost,0.8766 ± 0.0235,0.7165 ± 0.0747,0.6999 ± 0.0512,0.7015 ± 0.0535,0.9139 ± 0.0130,0.8788 ± 0.0184,0.8766 ± 0.0235,0.8765 ± 0.0209,0.9222 ± 0.0126,0.8780 ± 0.0065,0.7510 ± 0.0447,0.7210 ± 0.0370,0.7308 ± 0.0356,0.9182 ± 0.0137,0.8768 ± 0.0071,0.8780 ± 0.0065,0.8769 ± 0.0068,0.9232 ± 0.0095
svm,0.9000 ± 0.0126,0.6001 ± 0.0083,0.6177 ± 0.0084,0.6080 ± 0.0085,0.9475 ± 0.0087,0.8784 ± 0.0118,0.9000 ± 0.0126,0.8880 ± 0.0125,0.9576 ± 0.0072,0.8998 ± 0.0057,0.5998 ± 0.0039,0.6186 ± 0.0039,0.6089 ± 0.0039,0.9512 ± 0.0058,0.8746 ± 0.0055,0.8998 ± 0.0057,0.8868 ± 0.0057,0.9589 ± 0.0034
logistic_regression,0.8725 ± 0.0208,0.6667 ± 0.0314,0.6950 ± 0.0573,0.6720 ± 0.0348,0.9303 ± 0.0091,0.8835 ± 0.0115,0.8725 ± 0.0208,0.8749 ± 0.0177,0.9445 ± 0.0073,0.8814 ± 0.0070,0.7126 ± 0.0313,0.7397 ± 0.0310,0.7230 ± 0.0304,0.9329 ± 0.0093,0.8875 ± 0.0064,0.8814 ± 0.0070,0.8832 ± 0.0066,0.9466 ± 0.0051
knn,0.8722 ± 0.0116,0.6750 ± 0.1459,0.6193 ± 0.0324,0.6188 ± 0.0495,0.8627 ± 0.0313,0.8680 ± 0.0147,0.8722 ± 0.0116,0.8619 ± 0.0117,0.9315 ± 0.0076,0.8766 ± 0.0088,0.8034 ± 0.1601,0.6247 ± 0.0209,0.6290 ± 0.0350,0.8616 ± 0.0158,0.8773 ± 0.0174,0.8766 ± 0.0088,0.8654 ± 0.0099,0.9294 ± 0.0039
gaussian_nb,0.8055 ± 0.0266,0.6541 ± 0.0166,0.7924 ± 0.0382,0.6503 ± 0.0254,0.9038 ± 0.0278,0.8883 ± 0.0157,0.8055 ± 0.0266,0.8304 ± 0.0238,0.9271 ± 0.0119,0.8124 ± 0.0201,0.6585 ± 0.0151,0.7896 ± 0.0374,0.6633 ± 0.0247,0.8865 ± 0.0230,0.8818 ± 0.0089,0.8124 ± 0.0201,0.8358 ± 0.0157,0.9221 ± 0.0066


### NON MOTOR DOMAIN


In [23]:
domain_cols = NM_cols 
X_HY3_data_NM = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_HY3_full.csv", index_col=0)[domain_cols]
y_HY3_data_NM = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY3_full.csv", index_col=0)
print("X_HY3_data shape:", X_HY3_data_NM.shape)
print("y_HY3_data shape:", y_HY3_data_NM.shape)
X_HY3_data_NM.head()

X_HY3_data shape: (909, 65)
y_HY3_data shape: (909, 1)


,NP1COG_mean,NP1COG_min,NP1COG_max,NP1COG_var,NP1COG_std,NP1HALL_mean,NP1HALL_min,NP1HALL_max,NP1HALL_var,NP1HALL_std,...,NP1LTHD_mean,NP1LTHD_min,NP1LTHD_max,NP1LTHD_var,NP1LTHD_std,NP1FATG_mean,NP1FATG_min,NP1FATG_max,NP1FATG_var,NP1FATG_std
PATNO,,,,,,,,,,,,,,,,,,,,,
3003,0.000000,0.0,0.0,0.000000,0.00000,0.000000,0.0,0.0,0.000000,0.00000,...,1.333333,1.0,2.0,0.333333,0.57735,1.000000,1.0,1.0,0.000000,0.00000
3018,0.333333,0.0,1.0,0.333333,0.57735,0.000000,0.0,0.0,0.000000,0.00000,...,0.333333,0.0,1.0,0.333333,0.57735,1.000000,0.0,2.0,1.000000,1.00000
3020,1.000000,1.0,1.0,0.000000,0.00000,0.333333,0.0,1.0,0.333333,0.57735,...,0.333333,0.0,1.0,0.333333,0.57735,1.333333,1.0,2.0,0.333333,0.57735
3028,0.666667,0.0,1.0,0.333333,0.57735,0.000000,0.0,0.0,0.000000,0.00000,...,0.000000,0.0,0.0,0.000000,0.00000,0.666667,0.0,1.0,0.333333,0.57735
3051,0.000000,0.0,0.0,0.000000,0.00000,0.000000,0.0,0.0,0.000000,0.00000,...,0.000000,0.0,0.0,0.000000,0.00000,1.000000,1.0,1.0,0.000000,0.00000


In [24]:
df_HY3_NM = evaluate_models_10x10_oof_and_test(
    X_df=X_HY3_data_NM, 
    y_df=y_HY3_data_NM, 
    models=classification_models, 
    random_state=42
)

df_HY3_NM.to_csv(full_set_path_HY3 / "HY3_NM_StandardScaler.csv")

df_HY3_NM_min_max = evaluate_models_10x10_oof_and_test(
    X_df=X_HY3_data_NM, 
    y_df=y_HY3_data_NM, 
    models=classification_models,
    scaler = MinMaxScaler(), 
    random_state=42
)
df_HY3_NM_min_max.to_csv(full_set_path_HY3 / "HY3_NM_MinMaxScaler.csv")


Evaluating decision_tree...
Evaluating random_forest...
Evaluating extra_trees...
Evaluating xgboost...
Evaluating adaboost...
Evaluating svm...
Evaluating logistic_regression...
Evaluating knn...
Evaluating gaussian_nb...
Evaluating decision_tree...
Evaluating random_forest...
Evaluating extra_trees...
Evaluating xgboost...
Evaluating adaboost...
Evaluating svm...
Evaluating logistic_regression...
Evaluating knn...
Evaluating gaussian_nb...


In [25]:
df_HY3_NM.head(10)

,Accuracy_Testing,Precision_macro_Testing,Recall_macro_Testing,F1_macro_Testing,AUC_macro_Testing,Precision_weighted_Testing,Recall_weighted_Testing,F1_weighted_Testing,AUC_weighted_Testing,Accuracy_CV,Precision_macro_CV,Recall_macro_CV,F1_macro_CV,AUC_macro_CV,Precision_weighted_CV,Recall_weighted_CV,F1_weighted_CV,AUC_weighted_CV
decision_tree,0.5158 ± 0.0220,0.3711 ± 0.0301,0.3743 ± 0.0277,0.3711 ± 0.0277,0.5294 ± 0.0206,0.5183 ± 0.0241,0.5158 ± 0.0220,0.5162 ± 0.0222,0.5325 ± 0.0249,0.4961 ± 0.0227,0.3585 ± 0.0229,0.3585 ± 0.0237,0.3582 ± 0.0233,0.5153 ± 0.0197,0.4984 ± 0.0224,0.4961 ± 0.0227,0.4970 ± 0.0226,0.5149 ± 0.0243
random_forest,0.5652 ± 0.0201,0.3755 ± 0.0143,0.3806 ± 0.0140,0.3739 ± 0.0142,0.6162 ± 0.0344,0.5494 ± 0.0206,0.5652 ± 0.0201,0.5512 ± 0.0203,0.5901 ± 0.0234,0.5591 ± 0.0169,0.3712 ± 0.0121,0.3775 ± 0.0118,0.3708 ± 0.0120,0.6176 ± 0.0324,0.5418 ± 0.0173,0.5591 ± 0.0169,0.5453 ± 0.0171,0.5762 ± 0.0232
extra_trees,0.5557 ± 0.0126,0.3686 ± 0.0088,0.3755 ± 0.0086,0.3695 ± 0.0088,0.6146 ± 0.0337,0.5398 ± 0.0127,0.5557 ± 0.0126,0.5440 ± 0.0125,0.5861 ± 0.0208,0.5399 ± 0.0135,0.3572 ± 0.0095,0.3655 ± 0.0094,0.3596 ± 0.0094,0.6067 ± 0.0274,0.5221 ± 0.0136,0.5399 ± 0.0135,0.5284 ± 0.0135,0.5667 ± 0.0185
xgboost,0.5414 ± 0.0309,0.3612 ± 0.0195,0.3672 ± 0.0212,0.3631 ± 0.0204,0.5916 ± 0.0267,0.5294 ± 0.0282,0.5414 ± 0.0309,0.5339 ± 0.0295,0.5788 ± 0.0247,0.5384 ± 0.0216,0.3911 ± 0.0544,0.3751 ± 0.0221,0.3759 ± 0.0280,0.5960 ± 0.0301,0.5273 ± 0.0227,0.5384 ± 0.0216,0.5316 ± 0.0213,0.5663 ± 0.0244
adaboost,0.5553 ± 0.0317,0.3751 ± 0.0226,0.3776 ± 0.0251,0.3726 ± 0.0249,0.5973 ± 0.0315,0.5498 ± 0.0335,0.5553 ± 0.0317,0.5471 ± 0.0339,0.5399 ± 0.0226,0.5357 ± 0.0128,0.3820 ± 0.0195,0.3758 ± 0.0097,0.3757 ± 0.0116,0.6095 ± 0.0130,0.5273 ± 0.0124,0.5357 ± 0.0128,0.5300 ± 0.0130,0.5369 ± 0.0091
svm,0.5718 ± 0.0223,0.3795 ± 0.0154,0.3879 ± 0.0159,0.3827 ± 0.0159,0.6546 ± 0.0322,0.5559 ± 0.0223,0.5718 ± 0.0223,0.5624 ± 0.0227,0.6194 ± 0.0208,0.5553 ± 0.0141,0.3684 ± 0.0096,0.3775 ± 0.0094,0.3721 ± 0.0093,0.6236 ± 0.0196,0.5383 ± 0.0137,0.5553 ± 0.0141,0.5455 ± 0.0136,0.5799 ± 0.0143
logistic_regression,0.4930 ± 0.0251,0.3831 ± 0.0205,0.3822 ± 0.0506,0.3671 ± 0.0259,0.5579 ± 0.0366,0.5487 ± 0.0222,0.4930 ± 0.0251,0.5128 ± 0.0236,0.5784 ± 0.0230,0.4934 ± 0.0115,0.3808 ± 0.0094,0.3901 ± 0.0164,0.3700 ± 0.0097,0.5679 ± 0.0243,0.5389 ± 0.0133,0.4934 ± 0.0115,0.5080 ± 0.0120,0.5749 ± 0.0119
knn,0.5315 ± 0.0320,0.3537 ± 0.0215,0.3632 ± 0.0222,0.3578 ± 0.0219,0.5455 ± 0.0256,0.5190 ± 0.0313,0.5315 ± 0.0320,0.5244 ± 0.0319,0.5647 ± 0.0332,0.5143 ± 0.0241,0.3420 ± 0.0158,0.3521 ± 0.0161,0.3467 ± 0.0160,0.5353 ± 0.0198,0.5008 ± 0.0228,0.5143 ± 0.0241,0.5070 ± 0.0235,0.5406 ± 0.0228
gaussian_nb,0.4114 ± 0.0102,0.3647 ± 0.1199,0.5054 ± 0.0518,0.2685 ± 0.0150,0.6371 ± 0.0447,0.4966 ± 0.1830,0.4114 ± 0.0102,0.3046 ± 0.0150,0.5903 ± 0.0348,0.4077 ± 0.0089,0.4218 ± 0.0494,0.4944 ± 0.0292,0.2718 ± 0.0140,0.6168 ± 0.0299,0.5827 ± 0.0766,0.4077 ± 0.0089,0.3047 ± 0.0153,0.5685 ± 0.0173


In [26]:
df_HY3_NM_min_max.head(10)

,Accuracy_Testing,Precision_macro_Testing,Recall_macro_Testing,F1_macro_Testing,AUC_macro_Testing,Precision_weighted_Testing,Recall_weighted_Testing,F1_weighted_Testing,AUC_weighted_Testing,Accuracy_CV,Precision_macro_CV,Recall_macro_CV,F1_macro_CV,AUC_macro_CV,Precision_weighted_CV,Recall_weighted_CV,F1_weighted_CV,AUC_weighted_CV
decision_tree,0.5172 ± 0.0208,0.3722 ± 0.0294,0.3751 ± 0.0269,0.3722 ± 0.0270,0.5302 ± 0.0198,0.5193 ± 0.0229,0.5172 ± 0.0208,0.5175 ± 0.0210,0.5336 ± 0.0240,0.4953 ± 0.0244,0.3566 ± 0.0240,0.3562 ± 0.0245,0.3561 ± 0.0242,0.5137 ± 0.0209,0.4973 ± 0.0240,0.4953 ± 0.0244,0.4961 ± 0.0242,0.5138 ± 0.0260
random_forest,0.5667 ± 0.0219,0.3768 ± 0.0157,0.3816 ± 0.0148,0.3748 ± 0.0147,0.6174 ± 0.0341,0.5513 ± 0.0226,0.5667 ± 0.0219,0.5526 ± 0.0213,0.5896 ± 0.0243,0.5583 ± 0.0186,0.3705 ± 0.0131,0.3769 ± 0.0130,0.3703 ± 0.0132,0.6164 ± 0.0319,0.5409 ± 0.0188,0.5583 ± 0.0186,0.5445 ± 0.0188,0.5754 ± 0.0229
extra_trees,0.5557 ± 0.0126,0.3686 ± 0.0088,0.3755 ± 0.0086,0.3695 ± 0.0088,0.6146 ± 0.0337,0.5398 ± 0.0127,0.5557 ± 0.0126,0.5440 ± 0.0125,0.5861 ± 0.0208,0.5399 ± 0.0135,0.3572 ± 0.0095,0.3655 ± 0.0094,0.3596 ± 0.0094,0.6067 ± 0.0274,0.5221 ± 0.0136,0.5399 ± 0.0135,0.5284 ± 0.0135,0.5667 ± 0.0185
xgboost,0.5407 ± 0.0297,0.3607 ± 0.0186,0.3667 ± 0.0203,0.3626 ± 0.0195,0.5880 ± 0.0218,0.5287 ± 0.0270,0.5407 ± 0.0297,0.5331 ± 0.0283,0.5776 ± 0.0218,0.5384 ± 0.0214,0.3911 ± 0.0544,0.3750 ± 0.0220,0.3759 ± 0.0280,0.5962 ± 0.0313,0.5273 ± 0.0225,0.5384 ± 0.0214,0.5316 ± 0.0211,0.5667 ± 0.0248
adaboost,0.5553 ± 0.0317,0.3751 ± 0.0226,0.3776 ± 0.0251,0.3726 ± 0.0249,0.5973 ± 0.0315,0.5498 ± 0.0335,0.5553 ± 0.0317,0.5471 ± 0.0339,0.5399 ± 0.0226,0.5357 ± 0.0128,0.3820 ± 0.0195,0.3758 ± 0.0097,0.3757 ± 0.0116,0.6095 ± 0.0130,0.5273 ± 0.0124,0.5357 ± 0.0128,0.5300 ± 0.0130,0.5369 ± 0.0091
svm,0.5641 ± 0.0198,0.3743 ± 0.0139,0.3824 ± 0.0139,0.3772 ± 0.0140,0.6505 ± 0.0324,0.5482 ± 0.0200,0.5641 ± 0.0198,0.5545 ± 0.0201,0.6190 ± 0.0173,0.5524 ± 0.0135,0.3662 ± 0.0092,0.3755 ± 0.0090,0.3701 ± 0.0090,0.6225 ± 0.0190,0.5352 ± 0.0131,0.5524 ± 0.0135,0.5426 ± 0.0131,0.5831 ± 0.0131
logistic_regression,0.4755 ± 0.0171,0.3851 ± 0.0128,0.3950 ± 0.0317,0.3608 ± 0.0124,0.6075 ± 0.0225,0.5486 ± 0.0208,0.4755 ± 0.0171,0.4957 ± 0.0171,0.5825 ± 0.0167,0.4792 ± 0.0105,0.3905 ± 0.0109,0.4296 ± 0.0246,0.3735 ± 0.0115,0.6088 ± 0.0143,0.5442 ± 0.0135,0.4792 ± 0.0105,0.4947 ± 0.0114,0.5718 ± 0.0084
knn,0.5396 ± 0.0316,0.3600 ± 0.0207,0.3697 ± 0.0214,0.3636 ± 0.0214,0.5490 ± 0.0327,0.5285 ± 0.0304,0.5396 ± 0.0316,0.5322 ± 0.0316,0.5671 ± 0.0358,0.5146 ± 0.0161,0.3429 ± 0.0106,0.3532 ± 0.0109,0.3474 ± 0.0109,0.5394 ± 0.0174,0.5022 ± 0.0154,0.5146 ± 0.0161,0.5076 ± 0.0159,0.5473 ± 0.0161
gaussian_nb,0.4121 ± 0.0093,0.3553 ± 0.0955,0.4921 ± 0.0544,0.2693 ± 0.0153,0.6364 ± 0.0456,0.4839 ± 0.1466,0.4121 ± 0.0093,0.3097 ± 0.0213,0.5903 ± 0.0355,0.4113 ± 0.0116,0.4128 ± 0.0418,0.4932 ± 0.0259,0.2769 ± 0.0165,0.6163 ± 0.0300,0.5684 ± 0.0629,0.4113 ± 0.0116,0.3129 ± 0.0223,0.5684 ± 0.0169


### IMPACT MOTOR DOMAIN


In [27]:
domain_cols = IM_cols 
X_HY3_data_IM = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_HY3_full.csv", index_col=0)[domain_cols]
y_HY3_data_IM = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY3_full.csv", index_col=0)
print("X_HY3_data shape:", X_HY3_data_IM.shape)
print("y_HY3_data shape:", y_HY3_data_IM.shape)
X_HY3_data_IM.head()

X_HY3_data shape: (909, 65)
y_HY3_data shape: (909, 1)


,NP2SPCH_mean,NP2SPCH_min,NP2SPCH_max,NP2SPCH_var,NP2SPCH_std,NP2SALV_mean,NP2SALV_min,NP2SALV_max,NP2SALV_var,NP2SALV_std,...,NP2WALK_mean,NP2WALK_min,NP2WALK_max,NP2WALK_var,NP2WALK_std,NP2FREZ_mean,NP2FREZ_min,NP2FREZ_max,NP2FREZ_var,NP2FREZ_std
PATNO,,,,,,,,,,,,,,,,,,,,,
3003,0.666667,0.0,2.0,1.333333,1.154701,0.000000,0.0,0.0,0.000000,0.00000,...,1.000000,1.0,1.0,0.000000,0.00000,0.000000,0.0,0.0,0.000000,0.00000
3018,0.333333,0.0,1.0,0.333333,0.577350,2.000000,2.0,2.0,0.000000,0.00000,...,1.000000,1.0,1.0,0.000000,0.00000,0.000000,0.0,0.0,0.000000,0.00000
3020,0.333333,0.0,1.0,0.333333,0.577350,3.000000,3.0,3.0,0.000000,0.00000,...,0.666667,0.0,1.0,0.333333,0.57735,0.333333,0.0,1.0,0.333333,0.57735
3028,1.000000,1.0,1.0,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.00000,...,0.000000,0.0,0.0,0.000000,0.00000,0.000000,0.0,0.0,0.000000,0.00000
3051,1.000000,1.0,1.0,0.000000,0.000000,0.666667,0.0,1.0,0.333333,0.57735,...,0.000000,0.0,0.0,0.000000,0.00000,0.000000,0.0,0.0,0.000000,0.00000


In [28]:
df_HY3_IM = evaluate_models_10x10_oof_and_test(
    X_df=X_HY3_data_IM, 
    y_df=y_HY3_data_IM, 
    models=classification_models, 
    random_state=42
)

df_HY3_IM.to_csv(full_set_path_HY3 / "HY3_IM_StandardScaler.csv")

df_HY3_IM_min_max = evaluate_models_10x10_oof_and_test(
    X_df=X_HY3_data_IM, 
    y_df=y_HY3_data_IM, 
    models=classification_models,
    scaler = MinMaxScaler(), 
    random_state=42
)
df_HY3_IM_min_max.to_csv(full_set_path_HY3 / "HY3_IM_MinMaxScaler.csv")

Evaluating decision_tree...
Evaluating random_forest...
Evaluating extra_trees...
Evaluating xgboost...
Evaluating adaboost...
Evaluating svm...
Evaluating logistic_regression...
Evaluating knn...
Evaluating gaussian_nb...
Evaluating decision_tree...
Evaluating random_forest...
Evaluating extra_trees...
Evaluating xgboost...
Evaluating adaboost...
Evaluating svm...
Evaluating logistic_regression...
Evaluating knn...
Evaluating gaussian_nb...


In [29]:
df_HY3_IM.head(10)

,Accuracy_Testing,Precision_macro_Testing,Recall_macro_Testing,F1_macro_Testing,AUC_macro_Testing,Precision_weighted_Testing,Recall_weighted_Testing,F1_weighted_Testing,AUC_weighted_Testing,Accuracy_CV,Precision_macro_CV,Recall_macro_CV,F1_macro_CV,AUC_macro_CV,Precision_weighted_CV,Recall_weighted_CV,F1_weighted_CV,AUC_weighted_CV
decision_tree,0.7802 ± 0.0244,0.5994 ± 0.0477,0.6129 ± 0.0540,0.5974 ± 0.0423,0.7197 ± 0.0350,0.7921 ± 0.0239,0.7802 ± 0.0244,0.7837 ± 0.0235,0.7710 ± 0.0258,0.7923 ± 0.0127,0.6215 ± 0.0375,0.6219 ± 0.0396,0.6200 ± 0.0381,0.7283 ± 0.0243,0.7938 ± 0.0125,0.7923 ± 0.0127,0.7921 ± 0.0124,0.7805 ± 0.0154
random_forest,0.8377 ± 0.0102,0.5931 ± 0.1045,0.5770 ± 0.0156,0.5741 ± 0.0262,0.8838 ± 0.0218,0.8201 ± 0.0122,0.8377 ± 0.0102,0.8277 ± 0.0106,0.9080 ± 0.0062,0.8316 ± 0.0088,0.5557 ± 0.0059,0.5695 ± 0.0062,0.5622 ± 0.0061,0.8863 ± 0.0128,0.8088 ± 0.0086,0.8316 ± 0.0088,0.8196 ± 0.0087,0.8992 ± 0.0046
extra_trees,0.8260 ± 0.0154,0.6276 ± 0.1417,0.5829 ± 0.0380,0.5872 ± 0.0557,0.8771 ± 0.0252,0.8132 ± 0.0212,0.8260 ± 0.0154,0.8178 ± 0.0166,0.8902 ± 0.0095,0.8171 ± 0.0123,0.6198 ± 0.0909,0.5718 ± 0.0179,0.5748 ± 0.0281,0.8810 ± 0.0160,0.8031 ± 0.0155,0.8171 ± 0.0123,0.8079 ± 0.0129,0.8841 ± 0.0093
xgboost,0.8341 ± 0.0161,0.6018 ± 0.0638,0.5941 ± 0.0372,0.5952 ± 0.0469,0.8731 ± 0.0266,0.8224 ± 0.0173,0.8341 ± 0.0161,0.8277 ± 0.0162,0.8980 ± 0.0118,0.8214 ± 0.0071,0.6421 ± 0.0936,0.5916 ± 0.0305,0.6010 ± 0.0456,0.8555 ± 0.0162,0.8111 ± 0.0092,0.8214 ± 0.0071,0.8148 ± 0.0071,0.8818 ± 0.0094
adaboost,0.8300 ± 0.0207,0.6064 ± 0.0465,0.6146 ± 0.0613,0.6067 ± 0.0469,0.8468 ± 0.0235,0.8256 ± 0.0204,0.8300 ± 0.0207,0.8263 ± 0.0210,0.8388 ± 0.0246,0.8178 ± 0.0255,0.6450 ± 0.0458,0.6315 ± 0.0407,0.6349 ± 0.0402,0.8439 ± 0.0086,0.8160 ± 0.0231,0.8178 ± 0.0255,0.8162 ± 0.0247,0.8367 ± 0.0150
svm,0.8473 ± 0.0175,0.5648 ± 0.0116,0.5810 ± 0.0125,0.5723 ± 0.0119,0.8981 ± 0.0212,0.8267 ± 0.0173,0.8473 ± 0.0175,0.8361 ± 0.0172,0.9143 ± 0.0103,0.8379 ± 0.0082,0.5585 ± 0.0054,0.5754 ± 0.0060,0.5667 ± 0.0056,0.8936 ± 0.0157,0.8143 ± 0.0081,0.8379 ± 0.0082,0.8258 ± 0.0081,0.9047 ± 0.0064
logistic_regression,0.7949 ± 0.0152,0.6006 ± 0.0198,0.6388 ± 0.0452,0.6007 ± 0.0223,0.8323 ± 0.0280,0.8339 ± 0.0199,0.7949 ± 0.0152,0.8072 ± 0.0155,0.8731 ± 0.0131,0.7847 ± 0.0140,0.5995 ± 0.0182,0.6397 ± 0.0362,0.6052 ± 0.0245,0.8110 ± 0.0198,0.8158 ± 0.0100,0.7847 ± 0.0140,0.7964 ± 0.0115,0.8614 ± 0.0071
knn,0.8110 ± 0.0153,0.5751 ± 0.1052,0.5620 ± 0.0173,0.5562 ± 0.0275,0.7919 ± 0.0380,0.7970 ± 0.0164,0.8110 ± 0.0153,0.8008 ± 0.0152,0.8719 ± 0.0164,0.8044 ± 0.0095,0.6311 ± 0.1386,0.5609 ± 0.0114,0.5576 ± 0.0189,0.8079 ± 0.0186,0.7920 ± 0.0147,0.8044 ± 0.0095,0.7937 ± 0.0096,0.8669 ± 0.0118
gaussian_nb,0.6802 ± 0.0182,0.5667 ± 0.0137,0.6831 ± 0.0526,0.5507 ± 0.0173,0.8436 ± 0.0133,0.7634 ± 0.0153,0.6802 ± 0.0182,0.6857 ± 0.0172,0.8428 ± 0.0097,0.6766 ± 0.0252,0.5725 ± 0.0126,0.7048 ± 0.0232,0.5631 ± 0.0251,0.8471 ± 0.0124,0.7552 ± 0.0118,0.6766 ± 0.0252,0.6798 ± 0.0249,0.8377 ± 0.0078


In [30]:
df_HY3_IM_min_max.head(10)  

,Accuracy_Testing,Precision_macro_Testing,Recall_macro_Testing,F1_macro_Testing,AUC_macro_Testing,Precision_weighted_Testing,Recall_weighted_Testing,F1_weighted_Testing,AUC_weighted_Testing,Accuracy_CV,Precision_macro_CV,Recall_macro_CV,F1_macro_CV,AUC_macro_CV,Precision_weighted_CV,Recall_weighted_CV,F1_weighted_CV,AUC_weighted_CV
decision_tree,0.7813 ± 0.0231,0.6002 ± 0.0468,0.6137 ± 0.0531,0.5982 ± 0.0412,0.7207 ± 0.0332,0.7933 ± 0.0228,0.7813 ± 0.0231,0.7848 ± 0.0224,0.7725 ± 0.0239,0.7914 ± 0.0139,0.6184 ± 0.0378,0.6196 ± 0.0398,0.6172 ± 0.0381,0.7267 ± 0.0248,0.7931 ± 0.0126,0.7914 ± 0.0139,0.7913 ± 0.0132,0.7795 ± 0.0171
random_forest,0.8381 ± 0.0105,0.5935 ± 0.1043,0.5774 ± 0.0157,0.5745 ± 0.0261,0.8839 ± 0.0216,0.8207 ± 0.0125,0.8381 ± 0.0105,0.8282 ± 0.0109,0.9078 ± 0.0061,0.8311 ± 0.0080,0.5888 ± 0.1057,0.5709 ± 0.0083,0.5654 ± 0.0127,0.8862 ± 0.0127,0.8113 ± 0.0122,0.8311 ± 0.0080,0.8194 ± 0.0079,0.8990 ± 0.0047
extra_trees,0.8260 ± 0.0154,0.6276 ± 0.1417,0.5829 ± 0.0380,0.5872 ± 0.0557,0.8771 ± 0.0252,0.8132 ± 0.0212,0.8260 ± 0.0154,0.8178 ± 0.0166,0.8902 ± 0.0095,0.8171 ± 0.0123,0.6198 ± 0.0909,0.5718 ± 0.0179,0.5748 ± 0.0281,0.8810 ± 0.0160,0.8031 ± 0.0155,0.8171 ± 0.0123,0.8079 ± 0.0129,0.8841 ± 0.0093
xgboost,0.8341 ± 0.0161,0.6018 ± 0.0638,0.5941 ± 0.0372,0.5952 ± 0.0469,0.8731 ± 0.0266,0.8224 ± 0.0173,0.8341 ± 0.0161,0.8277 ± 0.0162,0.8980 ± 0.0118,0.8214 ± 0.0071,0.6421 ± 0.0936,0.5916 ± 0.0305,0.6010 ± 0.0456,0.8556 ± 0.0163,0.8111 ± 0.0092,0.8214 ± 0.0071,0.8148 ± 0.0071,0.8818 ± 0.0094
adaboost,0.8300 ± 0.0207,0.6064 ± 0.0465,0.6146 ± 0.0613,0.6067 ± 0.0469,0.8468 ± 0.0235,0.8256 ± 0.0204,0.8300 ± 0.0207,0.8263 ± 0.0210,0.8388 ± 0.0246,0.8178 ± 0.0255,0.6450 ± 0.0458,0.6315 ± 0.0407,0.6349 ± 0.0402,0.8439 ± 0.0086,0.8160 ± 0.0231,0.8178 ± 0.0255,0.8162 ± 0.0247,0.8366 ± 0.0150
svm,0.8542 ± 0.0160,0.5693 ± 0.0107,0.5859 ± 0.0113,0.5770 ± 0.0108,0.9018 ± 0.0211,0.8335 ± 0.0158,0.8542 ± 0.0160,0.8430 ± 0.0157,0.9174 ± 0.0104,0.8420 ± 0.0081,0.5612 ± 0.0053,0.5784 ± 0.0058,0.5695 ± 0.0055,0.8952 ± 0.0156,0.8183 ± 0.0080,0.8420 ± 0.0081,0.8298 ± 0.0079,0.9062 ± 0.0057
logistic_regression,0.7842 ± 0.0209,0.6155 ± 0.0226,0.7004 ± 0.0740,0.6141 ± 0.0275,0.8733 ± 0.0185,0.8441 ± 0.0191,0.7842 ± 0.0209,0.8015 ± 0.0189,0.8900 ± 0.0096,0.7851 ± 0.0149,0.6230 ± 0.0175,0.7111 ± 0.0395,0.6321 ± 0.0267,0.8632 ± 0.0129,0.8315 ± 0.0068,0.7851 ± 0.0149,0.7996 ± 0.0117,0.8789 ± 0.0057
knn,0.8168 ± 0.0159,0.6292 ± 0.1448,0.5749 ± 0.0260,0.5757 ± 0.0415,0.8044 ± 0.0443,0.8067 ± 0.0211,0.8168 ± 0.0159,0.8075 ± 0.0161,0.8781 ± 0.0167,0.8101 ± 0.0098,0.6320 ± 0.1482,0.5664 ± 0.0158,0.5640 ± 0.0278,0.8150 ± 0.0261,0.7971 ± 0.0142,0.8101 ± 0.0098,0.7994 ± 0.0091,0.8719 ± 0.0086
gaussian_nb,0.6835 ± 0.0157,0.5681 ± 0.0126,0.6852 ± 0.0537,0.5542 ± 0.0168,0.8437 ± 0.0132,0.7640 ± 0.0131,0.6835 ± 0.0157,0.6888 ± 0.0150,0.8426 ± 0.0101,0.6800 ± 0.0242,0.5746 ± 0.0128,0.7070 ± 0.0239,0.5666 ± 0.0240,0.8469 ± 0.0126,0.7570 ± 0.0125,0.6800 ± 0.0242,0.6835 ± 0.0238,0.8373 ± 0.0081


### MOTOR DOMAIN EXAM


In [31]:
domain_cols = M_cols 
X_HY3_data_M = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_HY3_full.csv", index_col=0)[domain_cols]
y_HY3_data_M = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY3_full.csv", index_col=0)
print("X_HY3_data shape:", X_HY3_data_M.shape)
print("y_HY3_data shape:", y_HY3_data_M.shape)
X_HY3_data_M.head()

X_HY3_data shape: (909, 165)
y_HY3_data shape: (909, 1)


,NP3SPCH_mean,NP3SPCH_min,NP3SPCH_max,NP3SPCH_var,NP3SPCH_std,NP3FACXP_mean,NP3FACXP_min,NP3FACXP_max,NP3FACXP_var,NP3FACXP_std,...,NP3RTALJ_mean,NP3RTALJ_min,NP3RTALJ_max,NP3RTALJ_var,NP3RTALJ_std,NP3RTCON_mean,NP3RTCON_min,NP3RTCON_max,NP3RTCON_var,NP3RTCON_std
PATNO,,,,,,,,,,,,,,,,,,,,,
3003,1.666667,1.0,2.0,0.333333,0.57735,1.666667,1.0,2.0,0.333333,0.57735,...,0.0,0.0,0.0,0.0,0.0,0.666667,0.0,1.0,0.333333,0.577350
3018,1.000000,1.0,1.0,0.000000,0.00000,1.666667,1.0,2.0,0.333333,0.57735,...,0.0,0.0,0.0,0.0,0.0,2.666667,1.0,4.0,2.333333,1.527525
3020,0.666667,0.0,1.0,0.333333,0.57735,1.666667,1.0,2.0,0.333333,0.57735,...,0.0,0.0,0.0,0.0,0.0,2.666667,2.0,4.0,1.333333,1.154701
3028,1.333333,1.0,2.0,0.333333,0.57735,2.000000,2.0,2.0,0.000000,0.00000,...,0.0,0.0,0.0,0.0,0.0,1.333333,0.0,3.0,2.333333,1.527525
3051,0.333333,0.0,1.0,0.333333,0.57735,1.000000,1.0,1.0,0.000000,0.00000,...,0.0,0.0,0.0,0.0,0.0,1.333333,1.0,2.0,0.333333,0.577350


In [32]:
df_HY3_M = evaluate_models_10x10_oof_and_test(
    X_df=X_HY3_data_M, 
    y_df=y_HY3_data_M, 
    models=classification_models, 
    random_state=42
)

df_HY3_M.to_csv(full_set_path_HY3 / "HY3_M_StandardScaler.csv")

df_HY3_M_min_max = evaluate_models_10x10_oof_and_test(
    X_df=X_HY3_data_M, 
    y_df=y_HY3_data_M, 
    models=classification_models,
    scaler = MinMaxScaler(), 
    random_state=42
)
df_HY3_M_min_max.to_csv(full_set_path_HY3 / "HY3_M_MinMaxScaler.csv")

Evaluating decision_tree...
Evaluating random_forest...
Evaluating extra_trees...
Evaluating xgboost...
Evaluating adaboost...
Evaluating svm...
Evaluating logistic_regression...
Evaluating knn...
Evaluating gaussian_nb...
Evaluating decision_tree...
Evaluating random_forest...
Evaluating extra_trees...
Evaluating xgboost...
Evaluating adaboost...
Evaluating svm...
Evaluating logistic_regression...
Evaluating knn...
Evaluating gaussian_nb...


## HY43


### FULL DOMAIN


In [33]:
domain_cols = SC_cols + NM_cols + IM_cols + M_cols
X_HY43_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_HY4_full.csv", index_col=0)[domain_cols]
y_HY43_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY4_3_full.csv", index_col=0)
print("X_HY43_data shape:", X_HY43_data.shape)
print("y_HY43_data shape:", y_HY43_data.shape)
X_HY43_data.head()

X_HY43_data shape: (909, 301)
y_HY43_data shape: (909, 1)


,ENRLLRRK2,ENRLGBA,SEX,RAWHITE,EDUCYRS,AGE_AT_VISIT,NP1COG_mean,NP1COG_min,NP1COG_max,NP1COG_var,...,NP3RTALJ_mean,NP3RTALJ_min,NP3RTALJ_max,NP3RTALJ_var,NP3RTALJ_std,NP3RTCON_mean,NP3RTCON_min,NP3RTCON_max,NP3RTCON_var,NP3RTCON_std
PATNO,,,,,,,,,,,,,,,,,,,,,
3003,0,0,0.0,1.0,16.0,59.7,0.000000,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.666667,0.0,1.0,0.333333,0.577350
3018,0,0,0.0,1.0,16.0,63.6,0.333333,0.0,1.0,0.333333,...,0.0,0.0,0.0,0.0,0.0,2.666667,1.0,4.0,2.333333,1.527525
3020,0,0,0.0,1.0,15.0,77.0,1.000000,1.0,1.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,2.666667,2.0,4.0,1.333333,1.154701
3028,0,0,1.0,1.0,22.0,78.8,0.666667,0.0,1.0,0.333333,...,0.0,0.0,0.0,0.0,0.0,1.333333,0.0,3.0,2.333333,1.527525
3051,0,0,1.0,1.0,18.0,74.7,0.000000,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,1.333333,1.0,2.0,0.333333,0.577350


In [34]:
df_HY43_full = evaluate_models_10x10_oof_and_test(
    X_df=X_HY43_data, 
    y_df=y_HY43_data, 
    models=classification_models, 
    random_state=42
)

df_HY43_full.to_csv(full_set_path_HY4 / "HY43_full_StandardScaler.csv")

df_HY43_full_min_max = evaluate_models_10x10_oof_and_test(
    X_df=X_HY43_data, 
    y_df=y_HY43_data, 
    models=classification_models,
    scaler = MinMaxScaler(), 
    random_state=42
)
df_HY43_full_min_max.to_csv(full_set_path_HY4 / "HY43_full_MinMaxScaler.csv")


Evaluating decision_tree...
Evaluating random_forest...
Evaluating extra_trees...
Evaluating xgboost...
Evaluating adaboost...
Evaluating svm...
Evaluating logistic_regression...
Evaluating knn...
Evaluating gaussian_nb...
Evaluating decision_tree...
Evaluating random_forest...
Evaluating extra_trees...
Evaluating xgboost...
Evaluating adaboost...
Evaluating svm...
Evaluating logistic_regression...
Evaluating knn...
Evaluating gaussian_nb...


In [35]:
df_HY43_full.head(10)

,Accuracy_Testing,Precision_macro_Testing,Recall_macro_Testing,F1_macro_Testing,AUC_macro_Testing,Precision_weighted_Testing,Recall_weighted_Testing,F1_weighted_Testing,AUC_weighted_Testing,Accuracy_CV,Precision_macro_CV,Recall_macro_CV,F1_macro_CV,AUC_macro_CV,Precision_weighted_CV,Recall_weighted_CV,F1_weighted_CV,AUC_weighted_CV
decision_tree,0.7516 ± 0.0229,0.6209 ± 0.0282,0.6206 ± 0.0354,0.6188 ± 0.0303,0.7480 ± 0.0226,0.7671 ± 0.0203,0.7516 ± 0.0229,0.7581 ± 0.0201,0.8132 ± 0.0168,0.7646 ± 0.0184,0.6329 ± 0.0221,0.6323 ± 0.0233,0.6322 ± 0.0228,0.7563 ± 0.0160,0.7733 ± 0.0159,0.7646 ± 0.0184,0.7687 ± 0.0168,0.8205 ± 0.0138
random_forest,0.8718 ± 0.0112,0.8817 ± 0.0728,0.6755 ± 0.0162,0.6656 ± 0.0252,0.8899 ± 0.0118,0.8756 ± 0.0261,0.8718 ± 0.0112,0.8346 ± 0.0141,0.9397 ± 0.0077,0.8627 ± 0.0062,0.7997 ± 0.0503,0.6597 ± 0.0117,0.6417 ± 0.0198,0.8926 ± 0.0050,0.8437 ± 0.0186,0.8627 ± 0.0062,0.8228 ± 0.0088,0.9392 ± 0.0022
extra_trees,0.8692 ± 0.0085,0.8543 ± 0.0684,0.6711 ± 0.0131,0.6584 ± 0.0227,0.8872 ± 0.0136,0.8653 ± 0.0221,0.8692 ± 0.0085,0.8311 ± 0.0105,0.9380 ± 0.0077,0.8594 ± 0.0041,0.8162 ± 0.0553,0.6587 ± 0.0079,0.6427 ± 0.0146,0.8908 ± 0.0064,0.8471 ± 0.0176,0.8594 ± 0.0041,0.8207 ± 0.0061,0.9382 ± 0.0040
xgboost,0.8520 ± 0.0146,0.7274 ± 0.0592,0.6690 ± 0.0176,0.6651 ± 0.0236,0.8888 ± 0.0106,0.8210 ± 0.0229,0.8520 ± 0.0146,0.8243 ± 0.0139,0.9374 ± 0.0066,0.8410 ± 0.0076,0.6856 ± 0.0373,0.6588 ± 0.0146,0.6538 ± 0.0211,0.8795 ± 0.0087,0.8042 ± 0.0142,0.8410 ± 0.0076,0.8158 ± 0.0092,0.9305 ± 0.0046
adaboost,0.8209 ± 0.0168,0.6582 ± 0.0367,0.6523 ± 0.0223,0.6495 ± 0.0279,0.8580 ± 0.0134,0.7945 ± 0.0188,0.8209 ± 0.0168,0.8049 ± 0.0165,0.9242 ± 0.0061,0.8148 ± 0.0110,0.6590 ± 0.0229,0.6540 ± 0.0167,0.6536 ± 0.0192,0.8464 ± 0.0070,0.7942 ± 0.0118,0.8148 ± 0.0110,0.8033 ± 0.0109,0.9172 ± 0.0040
svm,0.8612 ± 0.0072,0.7417 ± 0.1545,0.6517 ± 0.0066,0.6236 ± 0.0138,0.8658 ± 0.0128,0.8228 ± 0.0486,0.8612 ± 0.0072,0.8158 ± 0.0062,0.9256 ± 0.0074,0.8497 ± 0.0047,0.6680 ± 0.1172,0.6379 ± 0.0051,0.6058 ± 0.0085,0.8649 ± 0.0072,0.7923 ± 0.0390,0.8497 ± 0.0047,0.8036 ± 0.0051,0.9230 ± 0.0040
logistic_regression,0.7524 ± 0.0202,0.6264 ± 0.0255,0.6236 ± 0.0321,0.6223 ± 0.0280,0.8139 ± 0.0129,0.7771 ± 0.0162,0.7524 ± 0.0202,0.7629 ± 0.0176,0.8843 ± 0.0081,0.7469 ± 0.0108,0.6221 ± 0.0126,0.6200 ± 0.0149,0.6191 ± 0.0135,0.8093 ± 0.0145,0.7701 ± 0.0092,0.7469 ± 0.0108,0.7575 ± 0.0091,0.8803 ± 0.0090
knn,0.7897 ± 0.0268,0.6544 ± 0.0440,0.6271 ± 0.0317,0.6236 ± 0.0377,0.8364 ± 0.0205,0.7730 ± 0.0255,0.7897 ± 0.0268,0.7682 ± 0.0289,0.8992 ± 0.0164,0.7893 ± 0.0082,0.6415 ± 0.0261,0.6206 ± 0.0153,0.6151 ± 0.0199,0.8318 ± 0.0093,0.7667 ± 0.0121,0.7893 ± 0.0082,0.7661 ± 0.0090,0.8949 ± 0.0047
gaussian_nb,0.7158 ± 0.0337,0.6636 ± 0.0247,0.6535 ± 0.0376,0.6249 ± 0.0330,0.8477 ± 0.0172,0.8180 ± 0.0214,0.7158 ± 0.0337,0.7402 ± 0.0307,0.9076 ± 0.0138,0.6943 ± 0.0348,0.6424 ± 0.0142,0.6299 ± 0.0249,0.6039 ± 0.0284,0.8302 ± 0.0082,0.7971 ± 0.0120,0.6943 ± 0.0348,0.7211 ± 0.0322,0.8940 ± 0.0051


In [36]:
df_HY43_full_min_max.head(10)

,Accuracy_Testing,Precision_macro_Testing,Recall_macro_Testing,F1_macro_Testing,AUC_macro_Testing,Precision_weighted_Testing,Recall_weighted_Testing,F1_weighted_Testing,AUC_weighted_Testing,Accuracy_CV,Precision_macro_CV,Recall_macro_CV,F1_macro_CV,AUC_macro_CV,Precision_weighted_CV,Recall_weighted_CV,F1_weighted_CV,AUC_weighted_CV
decision_tree,0.7524 ± 0.0223,0.6219 ± 0.0292,0.6220 ± 0.0375,0.6198 ± 0.0313,0.7488 ± 0.0235,0.7676 ± 0.0207,0.7524 ± 0.0223,0.7586 ± 0.0195,0.8136 ± 0.0163,0.7646 ± 0.0181,0.6333 ± 0.0214,0.6327 ± 0.0225,0.6325 ± 0.0221,0.7565 ± 0.0155,0.7737 ± 0.0156,0.7646 ± 0.0181,0.7689 ± 0.0164,0.8206 ± 0.0135
random_forest,0.8718 ± 0.0108,0.8851 ± 0.0637,0.6755 ± 0.0155,0.6654 ± 0.0245,0.8903 ± 0.0121,0.8767 ± 0.0224,0.8718 ± 0.0108,0.8346 ± 0.0137,0.9398 ± 0.0079,0.8624 ± 0.0058,0.8048 ± 0.0628,0.6580 ± 0.0103,0.6384 ± 0.0175,0.8924 ± 0.0047,0.8452 ± 0.0223,0.8624 ± 0.0058,0.8217 ± 0.0080,0.9392 ± 0.0020
extra_trees,0.8692 ± 0.0085,0.8543 ± 0.0684,0.6711 ± 0.0131,0.6584 ± 0.0227,0.8872 ± 0.0136,0.8653 ± 0.0221,0.8692 ± 0.0085,0.8311 ± 0.0105,0.9380 ± 0.0077,0.8594 ± 0.0041,0.8162 ± 0.0553,0.6587 ± 0.0079,0.6427 ± 0.0146,0.8908 ± 0.0064,0.8471 ± 0.0176,0.8594 ± 0.0041,0.8207 ± 0.0061,0.9382 ± 0.0040
xgboost,0.8520 ± 0.0146,0.7274 ± 0.0592,0.6690 ± 0.0176,0.6651 ± 0.0236,0.8888 ± 0.0106,0.8210 ± 0.0229,0.8520 ± 0.0146,0.8243 ± 0.0139,0.9374 ± 0.0066,0.8410 ± 0.0076,0.6856 ± 0.0373,0.6588 ± 0.0146,0.6538 ± 0.0211,0.8795 ± 0.0086,0.8042 ± 0.0142,0.8410 ± 0.0076,0.8158 ± 0.0092,0.9305 ± 0.0046
adaboost,0.8209 ± 0.0168,0.6582 ± 0.0367,0.6523 ± 0.0223,0.6495 ± 0.0279,0.8580 ± 0.0134,0.7945 ± 0.0188,0.8209 ± 0.0168,0.8049 ± 0.0165,0.9242 ± 0.0061,0.8148 ± 0.0110,0.6590 ± 0.0229,0.6540 ± 0.0167,0.6536 ± 0.0192,0.8464 ± 0.0070,0.7942 ± 0.0118,0.8148 ± 0.0110,0.8033 ± 0.0109,0.9172 ± 0.0040
svm,0.8593 ± 0.0070,0.7080 ± 0.1509,0.6486 ± 0.0048,0.6187 ± 0.0108,0.8706 ± 0.0126,0.8113 ± 0.0478,0.8593 ± 0.0070,0.8136 ± 0.0047,0.9293 ± 0.0071,0.8508 ± 0.0051,0.6768 ± 0.1146,0.6387 ± 0.0046,0.6065 ± 0.0070,0.8719 ± 0.0071,0.7956 ± 0.0376,0.8508 ± 0.0051,0.8046 ± 0.0052,0.9276 ± 0.0038
logistic_regression,0.7648 ± 0.0210,0.6472 ± 0.0204,0.6454 ± 0.0275,0.6422 ± 0.0242,0.8547 ± 0.0144,0.7961 ± 0.0135,0.7648 ± 0.0210,0.7778 ± 0.0175,0.9171 ± 0.0068,0.7642 ± 0.0134,0.6458 ± 0.0176,0.6460 ± 0.0215,0.6423 ± 0.0187,0.8442 ± 0.0149,0.7942 ± 0.0139,0.7642 ± 0.0134,0.7771 ± 0.0132,0.9083 ± 0.0081
knn,0.8044 ± 0.0227,0.6594 ± 0.0487,0.6381 ± 0.0287,0.6345 ± 0.0353,0.8458 ± 0.0173,0.7832 ± 0.0222,0.8044 ± 0.0227,0.7840 ± 0.0223,0.9092 ± 0.0116,0.8038 ± 0.0103,0.6553 ± 0.0368,0.6337 ± 0.0195,0.6291 ± 0.0253,0.8417 ± 0.0089,0.7790 ± 0.0171,0.8038 ± 0.0103,0.7814 ± 0.0122,0.9060 ± 0.0048
gaussian_nb,0.7231 ± 0.0320,0.6643 ± 0.0284,0.6565 ± 0.0416,0.6304 ± 0.0339,0.8485 ± 0.0167,0.8178 ± 0.0246,0.7231 ± 0.0320,0.7472 ± 0.0287,0.9080 ± 0.0135,0.7003 ± 0.0311,0.6425 ± 0.0145,0.6319 ± 0.0251,0.6086 ± 0.0258,0.8311 ± 0.0085,0.7966 ± 0.0118,0.7003 ± 0.0311,0.7273 ± 0.0275,0.8945 ± 0.0054


### NON MOTOR DOMAIN


In [37]:
domain_cols = NM_cols 
X_HY43_data_NM = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_HY4_full.csv", index_col=0)[domain_cols]
y_HY43_data_NM = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY4_3_full.csv", index_col=0)
print("X_HY43_data shape:", X_HY43_data_NM.shape)
print("y_HY43_data shape:", y_HY43_data_NM.shape)
X_HY43_data_NM.head()

X_HY43_data shape: (909, 65)
y_HY43_data shape: (909, 1)


,NP1COG_mean,NP1COG_min,NP1COG_max,NP1COG_var,NP1COG_std,NP1HALL_mean,NP1HALL_min,NP1HALL_max,NP1HALL_var,NP1HALL_std,...,NP1LTHD_mean,NP1LTHD_min,NP1LTHD_max,NP1LTHD_var,NP1LTHD_std,NP1FATG_mean,NP1FATG_min,NP1FATG_max,NP1FATG_var,NP1FATG_std
PATNO,,,,,,,,,,,,,,,,,,,,,
3003,0.000000,0.0,0.0,0.000000,0.00000,0.000000,0.0,0.0,0.000000,0.00000,...,1.333333,1.0,2.0,0.333333,0.57735,1.000000,1.0,1.0,0.000000,0.00000
3018,0.333333,0.0,1.0,0.333333,0.57735,0.000000,0.0,0.0,0.000000,0.00000,...,0.333333,0.0,1.0,0.333333,0.57735,1.000000,0.0,2.0,1.000000,1.00000
3020,1.000000,1.0,1.0,0.000000,0.00000,0.333333,0.0,1.0,0.333333,0.57735,...,0.333333,0.0,1.0,0.333333,0.57735,1.333333,1.0,2.0,0.333333,0.57735
3028,0.666667,0.0,1.0,0.333333,0.57735,0.000000,0.0,0.0,0.000000,0.00000,...,0.000000,0.0,0.0,0.000000,0.00000,0.666667,0.0,1.0,0.333333,0.57735
3051,0.000000,0.0,0.0,0.000000,0.00000,0.000000,0.0,0.0,0.000000,0.00000,...,0.000000,0.0,0.0,0.000000,0.00000,1.000000,1.0,1.0,0.000000,0.00000


In [38]:
df_HY43_NM = evaluate_models_10x10_oof_and_test(
    X_df=X_HY43_data_NM, 
    y_df=y_HY43_data_NM, 
    models=classification_models, 
    random_state=42
)

df_HY43_NM.to_csv(full_set_path_HY4 / "HY43_NM_StandardScaler.csv")

df_HY43_NM_min_max = evaluate_models_10x10_oof_and_test(
    X_df=X_HY43_data_NM, 
    y_df=y_HY43_data_NM, 
    models=classification_models,
    scaler = MinMaxScaler(), 
    random_state=42
)
df_HY43_NM_min_max.to_csv(full_set_path_HY4 / "HY43_NM_MinMaxScaler.csv")


Evaluating decision_tree...
Evaluating random_forest...
Evaluating extra_trees...
Evaluating xgboost...
Evaluating adaboost...
Evaluating svm...
Evaluating logistic_regression...
Evaluating knn...
Evaluating gaussian_nb...
Evaluating decision_tree...
Evaluating random_forest...
Evaluating extra_trees...
Evaluating xgboost...
Evaluating adaboost...
Evaluating svm...
Evaluating logistic_regression...
Evaluating knn...
Evaluating gaussian_nb...


In [39]:
df_HY43_NM.head(10)

,Accuracy_Testing,Precision_macro_Testing,Recall_macro_Testing,F1_macro_Testing,AUC_macro_Testing,Precision_weighted_Testing,Recall_weighted_Testing,F1_weighted_Testing,AUC_weighted_Testing,Accuracy_CV,Precision_macro_CV,Recall_macro_CV,F1_macro_CV,AUC_macro_CV,Precision_weighted_CV,Recall_weighted_CV,F1_weighted_CV,AUC_weighted_CV
decision_tree,0.4473 ± 0.0287,0.3624 ± 0.0312,0.3609 ± 0.0310,0.3606 ± 0.0309,0.5193 ± 0.0231,0.4511 ± 0.0293,0.4473 ± 0.0287,0.4481 ± 0.0284,0.5294 ± 0.0253,0.4363 ± 0.0225,0.3611 ± 0.0228,0.3613 ± 0.0242,0.3608 ± 0.0234,0.5177 ± 0.0191,0.4420 ± 0.0219,0.4363 ± 0.0225,0.4389 ± 0.0220,0.5215 ± 0.0198
random_forest,0.5238 ± 0.0180,0.3702 ± 0.0503,0.3931 ± 0.0132,0.3734 ± 0.0131,0.5968 ± 0.0312,0.4775 ± 0.0208,0.5238 ± 0.0180,0.4965 ± 0.0173,0.6077 ± 0.0240,0.5024 ± 0.0137,0.3632 ± 0.0214,0.3790 ± 0.0105,0.3636 ± 0.0116,0.5836 ± 0.0145,0.4625 ± 0.0136,0.5024 ± 0.0137,0.4790 ± 0.0135,0.5932 ± 0.0150
extra_trees,0.5084 ± 0.0183,0.3557 ± 0.0330,0.3825 ± 0.0160,0.3646 ± 0.0186,0.5759 ± 0.0209,0.4634 ± 0.0234,0.5084 ± 0.0183,0.4831 ± 0.0193,0.5900 ± 0.0166,0.5036 ± 0.0165,0.3606 ± 0.0179,0.3796 ± 0.0119,0.3640 ± 0.0115,0.5760 ± 0.0220,0.4632 ± 0.0145,0.5036 ± 0.0165,0.4803 ± 0.0152,0.5866 ± 0.0196
xgboost,0.4945 ± 0.0265,0.3778 ± 0.0256,0.3820 ± 0.0193,0.3751 ± 0.0201,0.5782 ± 0.0237,0.4681 ± 0.0238,0.4945 ± 0.0265,0.4791 ± 0.0250,0.5878 ± 0.0234,0.4863 ± 0.0194,0.3743 ± 0.0216,0.3750 ± 0.0147,0.3687 ± 0.0147,0.5627 ± 0.0149,0.4618 ± 0.0173,0.4863 ± 0.0194,0.4716 ± 0.0175,0.5721 ± 0.0183
adaboost,0.5352 ± 0.0227,0.3580 ± 0.0163,0.4007 ± 0.0171,0.3762 ± 0.0169,0.5850 ± 0.0192,0.4780 ± 0.0217,0.5352 ± 0.0227,0.5023 ± 0.0225,0.6029 ± 0.0190,0.5226 ± 0.0176,0.3487 ± 0.0118,0.3906 ± 0.0131,0.3675 ± 0.0124,0.5704 ± 0.0200,0.4664 ± 0.0157,0.5226 ± 0.0176,0.4916 ± 0.0166,0.5882 ± 0.0214
svm,0.5377 ± 0.0229,0.3592 ± 0.0152,0.4026 ± 0.0172,0.3782 ± 0.0164,0.5950 ± 0.0225,0.4796 ± 0.0203,0.5377 ± 0.0229,0.5051 ± 0.0219,0.6213 ± 0.0211,0.5269 ± 0.0127,0.3519 ± 0.0087,0.3937 ± 0.0095,0.3705 ± 0.0089,0.5720 ± 0.0184,0.4705 ± 0.0117,0.5269 ± 0.0127,0.4957 ± 0.0119,0.5979 ± 0.0135
logistic_regression,0.4319 ± 0.0306,0.4071 ± 0.0202,0.4045 ± 0.0251,0.3872 ± 0.0231,0.5814 ± 0.0137,0.4996 ± 0.0249,0.4319 ± 0.0306,0.4547 ± 0.0284,0.5988 ± 0.0145,0.4226 ± 0.0197,0.3949 ± 0.0145,0.3894 ± 0.0188,0.3765 ± 0.0161,0.5722 ± 0.0108,0.4880 ± 0.0177,0.4226 ± 0.0197,0.4467 ± 0.0189,0.5883 ± 0.0153
knn,0.4685 ± 0.0264,0.3392 ± 0.0278,0.3547 ± 0.0199,0.3372 ± 0.0204,0.5489 ± 0.0252,0.4360 ± 0.0268,0.4685 ± 0.0264,0.4421 ± 0.0251,0.5621 ± 0.0256,0.4690 ± 0.0155,0.3457 ± 0.0123,0.3558 ± 0.0098,0.3408 ± 0.0100,0.5332 ± 0.0148,0.4382 ± 0.0135,0.4690 ± 0.0155,0.4445 ± 0.0157,0.5430 ± 0.0172
gaussian_nb,0.4952 ± 0.0157,0.3794 ± 0.0421,0.3725 ± 0.0143,0.3209 ± 0.0215,0.5976 ± 0.0231,0.4912 ± 0.0333,0.4952 ± 0.0157,0.4236 ± 0.0221,0.6152 ± 0.0282,0.4912 ± 0.0183,0.3756 ± 0.0258,0.3717 ± 0.0097,0.3265 ± 0.0156,0.5715 ± 0.0087,0.4831 ± 0.0164,0.4912 ± 0.0183,0.4271 ± 0.0179,0.5932 ± 0.0110


In [40]:
df_HY43_NM_min_max.head(10)

,Accuracy_Testing,Precision_macro_Testing,Recall_macro_Testing,F1_macro_Testing,AUC_macro_Testing,Precision_weighted_Testing,Recall_weighted_Testing,F1_weighted_Testing,AUC_weighted_Testing,Accuracy_CV,Precision_macro_CV,Recall_macro_CV,F1_macro_CV,AUC_macro_CV,Precision_weighted_CV,Recall_weighted_CV,F1_weighted_CV,AUC_weighted_CV
decision_tree,0.4458 ± 0.0301,0.3617 ± 0.0323,0.3598 ± 0.0326,0.3597 ± 0.0323,0.5184 ± 0.0241,0.4500 ± 0.0306,0.4458 ± 0.0301,0.4468 ± 0.0298,0.5283 ± 0.0263,0.4366 ± 0.0214,0.3617 ± 0.0218,0.3619 ± 0.0231,0.3613 ± 0.0224,0.5181 ± 0.0182,0.4425 ± 0.0206,0.4366 ± 0.0214,0.4393 ± 0.0208,0.5219 ± 0.0187
random_forest,0.5242 ± 0.0181,0.3703 ± 0.0553,0.3934 ± 0.0143,0.3735 ± 0.0150,0.5969 ± 0.0315,0.4776 ± 0.0252,0.5242 ± 0.0181,0.4966 ± 0.0173,0.6077 ± 0.0243,0.5025 ± 0.0148,0.3621 ± 0.0212,0.3788 ± 0.0113,0.3630 ± 0.0121,0.5840 ± 0.0140,0.4623 ± 0.0142,0.5025 ± 0.0148,0.4789 ± 0.0146,0.5932 ± 0.0147
extra_trees,0.5084 ± 0.0183,0.3557 ± 0.0330,0.3825 ± 0.0160,0.3646 ± 0.0186,0.5759 ± 0.0209,0.4634 ± 0.0234,0.5084 ± 0.0183,0.4831 ± 0.0193,0.5900 ± 0.0166,0.5036 ± 0.0165,0.3606 ± 0.0179,0.3796 ± 0.0119,0.3640 ± 0.0115,0.5760 ± 0.0220,0.4632 ± 0.0145,0.5036 ± 0.0165,0.4803 ± 0.0152,0.5865 ± 0.0196
xgboost,0.4945 ± 0.0265,0.3778 ± 0.0256,0.3820 ± 0.0193,0.3751 ± 0.0201,0.5782 ± 0.0237,0.4681 ± 0.0238,0.4945 ± 0.0265,0.4791 ± 0.0250,0.5878 ± 0.0234,0.4869 ± 0.0204,0.3747 ± 0.0216,0.3754 ± 0.0152,0.3691 ± 0.0151,0.5626 ± 0.0148,0.4621 ± 0.0180,0.4869 ± 0.0204,0.4721 ± 0.0184,0.5719 ± 0.0183
adaboost,0.5352 ± 0.0227,0.3580 ± 0.0163,0.4007 ± 0.0171,0.3762 ± 0.0169,0.5850 ± 0.0192,0.4780 ± 0.0217,0.5352 ± 0.0227,0.5023 ± 0.0225,0.6029 ± 0.0190,0.5226 ± 0.0176,0.3487 ± 0.0118,0.3906 ± 0.0131,0.3675 ± 0.0124,0.5703 ± 0.0202,0.4664 ± 0.0157,0.5226 ± 0.0176,0.4916 ± 0.0166,0.5881 ± 0.0216
svm,0.5377 ± 0.0127,0.3595 ± 0.0083,0.4025 ± 0.0095,0.3777 ± 0.0097,0.5898 ± 0.0173,0.4799 ± 0.0110,0.5377 ± 0.0127,0.5044 ± 0.0129,0.6196 ± 0.0189,0.5327 ± 0.0129,0.3560 ± 0.0087,0.3980 ± 0.0096,0.3743 ± 0.0091,0.5692 ± 0.0176,0.4760 ± 0.0117,0.5327 ± 0.0129,0.5007 ± 0.0122,0.5988 ± 0.0148
logistic_regression,0.4381 ± 0.0260,0.4196 ± 0.0188,0.4159 ± 0.0180,0.3956 ± 0.0182,0.5900 ± 0.0184,0.5152 ± 0.0260,0.4381 ± 0.0260,0.4635 ± 0.0250,0.6109 ± 0.0194,0.4234 ± 0.0229,0.4015 ± 0.0129,0.3884 ± 0.0185,0.3771 ± 0.0172,0.5781 ± 0.0139,0.4995 ± 0.0152,0.4234 ± 0.0229,0.4507 ± 0.0206,0.5962 ± 0.0152
knn,0.4868 ± 0.0258,0.3753 ± 0.0440,0.3743 ± 0.0252,0.3608 ± 0.0285,0.5552 ± 0.0246,0.4619 ± 0.0326,0.4868 ± 0.0258,0.4621 ± 0.0246,0.5658 ± 0.0218,0.4778 ± 0.0122,0.3738 ± 0.0219,0.3668 ± 0.0107,0.3549 ± 0.0135,0.5407 ± 0.0113,0.4530 ± 0.0134,0.4778 ± 0.0122,0.4541 ± 0.0134,0.5473 ± 0.0142
gaussian_nb,0.4996 ± 0.0149,0.3810 ± 0.0350,0.3767 ± 0.0129,0.3289 ± 0.0205,0.5985 ± 0.0225,0.4934 ± 0.0286,0.4996 ± 0.0149,0.4328 ± 0.0211,0.6156 ± 0.0280,0.4942 ± 0.0182,0.3796 ± 0.0228,0.3747 ± 0.0096,0.3335 ± 0.0149,0.5715 ± 0.0087,0.4856 ± 0.0152,0.4942 ± 0.0182,0.4351 ± 0.0163,0.5930 ± 0.0108


### IMPACT MOTOR DOMAIN


In [41]:
domain_cols = IM_cols 
X_HY43_data_IM = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_HY4_full.csv", index_col=0)[domain_cols]
y_HY43_data_IM = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY4_3_full.csv", index_col=0)
print("X_HY43_data shape:", X_HY43_data_IM.shape)
print("y_HY43_data shape:", y_HY43_data_IM.shape)
X_HY43_data_IM.head()

X_HY43_data shape: (909, 65)
y_HY43_data shape: (909, 1)


,NP2SPCH_mean,NP2SPCH_min,NP2SPCH_max,NP2SPCH_var,NP2SPCH_std,NP2SALV_mean,NP2SALV_min,NP2SALV_max,NP2SALV_var,NP2SALV_std,...,NP2WALK_mean,NP2WALK_min,NP2WALK_max,NP2WALK_var,NP2WALK_std,NP2FREZ_mean,NP2FREZ_min,NP2FREZ_max,NP2FREZ_var,NP2FREZ_std
PATNO,,,,,,,,,,,,,,,,,,,,,
3003,0.666667,0.0,2.0,1.333333,1.154701,0.000000,0.0,0.0,0.000000,0.00000,...,1.000000,1.0,1.0,0.000000,0.00000,0.000000,0.0,0.0,0.000000,0.00000
3018,0.333333,0.0,1.0,0.333333,0.577350,2.000000,2.0,2.0,0.000000,0.00000,...,1.000000,1.0,1.0,0.000000,0.00000,0.000000,0.0,0.0,0.000000,0.00000
3020,0.333333,0.0,1.0,0.333333,0.577350,3.000000,3.0,3.0,0.000000,0.00000,...,0.666667,0.0,1.0,0.333333,0.57735,0.333333,0.0,1.0,0.333333,0.57735
3028,1.000000,1.0,1.0,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.00000,...,0.000000,0.0,0.0,0.000000,0.00000,0.000000,0.0,0.0,0.000000,0.00000
3051,1.000000,1.0,1.0,0.000000,0.000000,0.666667,0.0,1.0,0.333333,0.57735,...,0.000000,0.0,0.0,0.000000,0.00000,0.000000,0.0,0.0,0.000000,0.00000


In [42]:
df_HY43_IM = evaluate_models_10x10_oof_and_test(
    X_df=X_HY43_data_IM, 
    y_df=y_HY43_data_IM, 
    models=classification_models, 
    random_state=42
)

df_HY43_IM.to_csv(full_set_path_HY4 / "HY43_IM_StandardScaler.csv")

df_HY43_IM_min_max = evaluate_models_10x10_oof_and_test(
    X_df=X_HY43_data_IM, 
    y_df=y_HY43_data_IM, 
    models=classification_models,
    scaler = MinMaxScaler(), 
    random_state=42
)
df_HY43_IM_min_max.to_csv(full_set_path_HY4 / "HY43_IM_MinMaxScaler.csv")


Evaluating decision_tree...
Evaluating random_forest...
Evaluating extra_trees...
Evaluating xgboost...
Evaluating adaboost...
Evaluating svm...
Evaluating logistic_regression...
Evaluating knn...
Evaluating gaussian_nb...
Evaluating decision_tree...
Evaluating random_forest...
Evaluating extra_trees...
Evaluating xgboost...
Evaluating adaboost...
Evaluating svm...
Evaluating logistic_regression...
Evaluating knn...
Evaluating gaussian_nb...


In [43]:
df_HY43_IM.head(10)

,Accuracy_Testing,Precision_macro_Testing,Recall_macro_Testing,F1_macro_Testing,AUC_macro_Testing,Precision_weighted_Testing,Recall_weighted_Testing,F1_weighted_Testing,AUC_weighted_Testing,Accuracy_CV,Precision_macro_CV,Recall_macro_CV,F1_macro_CV,AUC_macro_CV,Precision_weighted_CV,Recall_weighted_CV,F1_weighted_CV,AUC_weighted_CV
decision_tree,0.6996 ± 0.0319,0.5550 ± 0.0350,0.5556 ± 0.0341,0.5535 ± 0.0339,0.6655 ± 0.0269,0.7007 ± 0.0272,0.6996 ± 0.0319,0.6985 ± 0.0279,0.7282 ± 0.0275,0.6984 ± 0.0184,0.5511 ± 0.0215,0.5526 ± 0.0201,0.5514 ± 0.0209,0.6611 ± 0.0183,0.6945 ± 0.0187,0.6984 ± 0.0184,0.6960 ± 0.0184,0.7250 ± 0.0196
random_forest,0.7835 ± 0.0226,0.5946 ± 0.0519,0.5994 ± 0.0249,0.5854 ± 0.0301,0.7799 ± 0.0162,0.7401 ± 0.0278,0.7835 ± 0.0226,0.7564 ± 0.0232,0.8636 ± 0.0140,0.7675 ± 0.0135,0.5628 ± 0.0207,0.5820 ± 0.0118,0.5648 ± 0.0132,0.7692 ± 0.0116,0.7193 ± 0.0135,0.7675 ± 0.0135,0.7393 ± 0.0121,0.8514 ± 0.0091
extra_trees,0.7667 ± 0.0150,0.5603 ± 0.0210,0.5826 ± 0.0123,0.5659 ± 0.0149,0.7762 ± 0.0195,0.7190 ± 0.0166,0.7667 ± 0.0150,0.7396 ± 0.0149,0.8495 ± 0.0173,0.7591 ± 0.0149,0.5601 ± 0.0313,0.5758 ± 0.0134,0.5590 ± 0.0152,0.7718 ± 0.0117,0.7117 ± 0.0155,0.7591 ± 0.0149,0.7311 ± 0.0136,0.8424 ± 0.0113
xgboost,0.7714 ± 0.0216,0.5941 ± 0.0294,0.5978 ± 0.0179,0.5888 ± 0.0200,0.7688 ± 0.0181,0.7350 ± 0.0163,0.7714 ± 0.0216,0.7502 ± 0.0170,0.8542 ± 0.0147,0.7552 ± 0.0086,0.5817 ± 0.0172,0.5847 ± 0.0109,0.5768 ± 0.0138,0.7535 ± 0.0141,0.7205 ± 0.0113,0.7552 ± 0.0086,0.7352 ± 0.0101,0.8366 ± 0.0110
adaboost,0.8084 ± 0.0221,0.5974 ± 0.1092,0.6080 ± 0.0181,0.5784 ± 0.0194,0.8125 ± 0.0127,0.7447 ± 0.0418,0.8084 ± 0.0221,0.7662 ± 0.0205,0.8837 ± 0.0112,0.7962 ± 0.0084,0.5846 ± 0.0674,0.5988 ± 0.0093,0.5713 ± 0.0134,0.7956 ± 0.0090,0.7339 ± 0.0250,0.7962 ± 0.0084,0.7561 ± 0.0093,0.8686 ± 0.0049
svm,0.8073 ± 0.0192,0.5392 ± 0.0136,0.6047 ± 0.0144,0.5697 ± 0.0137,0.7986 ± 0.0189,0.7201 ± 0.0182,0.8073 ± 0.0192,0.7607 ± 0.0184,0.8715 ± 0.0166,0.7925 ± 0.0095,0.5293 ± 0.0062,0.5926 ± 0.0071,0.5589 ± 0.0066,0.7834 ± 0.0103,0.7081 ± 0.0083,0.7925 ± 0.0095,0.7475 ± 0.0089,0.8577 ± 0.0065
logistic_regression,0.6901 ± 0.0235,0.6094 ± 0.0207,0.5985 ± 0.0272,0.5892 ± 0.0223,0.7986 ± 0.0194,0.7571 ± 0.0201,0.6901 ± 0.0235,0.7126 ± 0.0213,0.8597 ± 0.0171,0.6814 ± 0.0132,0.5991 ± 0.0101,0.5929 ± 0.0155,0.5829 ± 0.0129,0.7855 ± 0.0144,0.7425 ± 0.0081,0.6814 ± 0.0132,0.7030 ± 0.0110,0.8469 ± 0.0101
knn,0.7484 ± 0.0209,0.5610 ± 0.0604,0.5694 ± 0.0211,0.5510 ± 0.0267,0.7584 ± 0.0154,0.7025 ± 0.0269,0.7484 ± 0.0209,0.7168 ± 0.0196,0.8382 ± 0.0125,0.7332 ± 0.0094,0.5504 ± 0.0313,0.5583 ± 0.0100,0.5422 ± 0.0143,0.7428 ± 0.0172,0.6900 ± 0.0128,0.7332 ± 0.0094,0.7040 ± 0.0083,0.8247 ± 0.0131
gaussian_nb,0.6007 ± 0.0228,0.5776 ± 0.0261,0.5371 ± 0.0324,0.5124 ± 0.0270,0.7866 ± 0.0209,0.7234 ± 0.0288,0.6007 ± 0.0228,0.6170 ± 0.0268,0.8518 ± 0.0192,0.5895 ± 0.0152,0.5698 ± 0.0156,0.5388 ± 0.0194,0.5067 ± 0.0167,0.7728 ± 0.0115,0.7108 ± 0.0171,0.5895 ± 0.0152,0.6052 ± 0.0167,0.8351 ± 0.0089


In [44]:
df_HY43_IM_min_max.head(10)

,Accuracy_Testing,Precision_macro_Testing,Recall_macro_Testing,F1_macro_Testing,AUC_macro_Testing,Precision_weighted_Testing,Recall_weighted_Testing,F1_weighted_Testing,AUC_weighted_Testing,Accuracy_CV,Precision_macro_CV,Recall_macro_CV,F1_macro_CV,AUC_macro_CV,Precision_weighted_CV,Recall_weighted_CV,F1_weighted_CV,AUC_weighted_CV
decision_tree,0.7000 ± 0.0324,0.5551 ± 0.0351,0.5559 ± 0.0338,0.5536 ± 0.0339,0.6649 ± 0.0276,0.7002 ± 0.0271,0.7000 ± 0.0324,0.6983 ± 0.0280,0.7273 ± 0.0280,0.6981 ± 0.0193,0.5506 ± 0.0214,0.5524 ± 0.0203,0.5511 ± 0.0209,0.6607 ± 0.0189,0.6937 ± 0.0188,0.6981 ± 0.0193,0.6955 ± 0.0189,0.7245 ± 0.0204
random_forest,0.7835 ± 0.0233,0.5944 ± 0.0519,0.5994 ± 0.0256,0.5855 ± 0.0305,0.7796 ± 0.0164,0.7400 ± 0.0278,0.7835 ± 0.0233,0.7565 ± 0.0237,0.8634 ± 0.0139,0.7653 ± 0.0138,0.5590 ± 0.0201,0.5800 ± 0.0118,0.5627 ± 0.0132,0.7687 ± 0.0118,0.7170 ± 0.0134,0.7653 ± 0.0138,0.7374 ± 0.0122,0.8511 ± 0.0093
extra_trees,0.7667 ± 0.0150,0.5603 ± 0.0210,0.5826 ± 0.0123,0.5659 ± 0.0149,0.7762 ± 0.0195,0.7190 ± 0.0166,0.7667 ± 0.0150,0.7396 ± 0.0149,0.8495 ± 0.0173,0.7591 ± 0.0149,0.5601 ± 0.0313,0.5758 ± 0.0134,0.5590 ± 0.0152,0.7718 ± 0.0117,0.7117 ± 0.0155,0.7591 ± 0.0149,0.7311 ± 0.0136,0.8424 ± 0.0113
xgboost,0.7714 ± 0.0216,0.5941 ± 0.0294,0.5978 ± 0.0179,0.5888 ± 0.0200,0.7688 ± 0.0181,0.7350 ± 0.0163,0.7714 ± 0.0216,0.7502 ± 0.0170,0.8542 ± 0.0147,0.7552 ± 0.0086,0.5817 ± 0.0172,0.5847 ± 0.0109,0.5768 ± 0.0138,0.7535 ± 0.0141,0.7205 ± 0.0113,0.7552 ± 0.0086,0.7352 ± 0.0101,0.8366 ± 0.0110
adaboost,0.8084 ± 0.0221,0.5974 ± 0.1092,0.6080 ± 0.0181,0.5784 ± 0.0194,0.8124 ± 0.0127,0.7447 ± 0.0418,0.8084 ± 0.0221,0.7662 ± 0.0205,0.8837 ± 0.0113,0.7962 ± 0.0084,0.5846 ± 0.0674,0.5988 ± 0.0093,0.5713 ± 0.0134,0.7956 ± 0.0090,0.7339 ± 0.0250,0.7962 ± 0.0084,0.7561 ± 0.0093,0.8686 ± 0.0049
svm,0.8114 ± 0.0183,0.5416 ± 0.0131,0.6077 ± 0.0137,0.5725 ± 0.0131,0.7985 ± 0.0173,0.7232 ± 0.0175,0.8114 ± 0.0183,0.7644 ± 0.0175,0.8734 ± 0.0155,0.7980 ± 0.0087,0.5325 ± 0.0056,0.5967 ± 0.0065,0.5626 ± 0.0061,0.7870 ± 0.0081,0.7123 ± 0.0074,0.7980 ± 0.0087,0.7524 ± 0.0082,0.8609 ± 0.0055
logistic_regression,0.6967 ± 0.0250,0.6097 ± 0.0193,0.5976 ± 0.0296,0.5900 ± 0.0254,0.8129 ± 0.0164,0.7586 ± 0.0162,0.6967 ± 0.0250,0.7165 ± 0.0213,0.8794 ± 0.0132,0.6903 ± 0.0116,0.6073 ± 0.0105,0.5983 ± 0.0156,0.5880 ± 0.0125,0.8015 ± 0.0114,0.7547 ± 0.0090,0.6903 ± 0.0116,0.7113 ± 0.0101,0.8661 ± 0.0088
knn,0.7476 ± 0.0214,0.5713 ± 0.0512,0.5714 ± 0.0219,0.5560 ± 0.0263,0.7675 ± 0.0169,0.7064 ± 0.0256,0.7476 ± 0.0214,0.7181 ± 0.0211,0.8455 ± 0.0150,0.7403 ± 0.0088,0.5618 ± 0.0251,0.5658 ± 0.0104,0.5508 ± 0.0158,0.7480 ± 0.0206,0.6979 ± 0.0126,0.7403 ± 0.0088,0.7111 ± 0.0105,0.8307 ± 0.0160
gaussian_nb,0.6044 ± 0.0216,0.5761 ± 0.0247,0.5374 ± 0.0315,0.5142 ± 0.0259,0.7862 ± 0.0208,0.7212 ± 0.0272,0.6044 ± 0.0216,0.6198 ± 0.0252,0.8512 ± 0.0194,0.5921 ± 0.0153,0.5668 ± 0.0144,0.5357 ± 0.0176,0.5065 ± 0.0154,0.7731 ± 0.0114,0.7077 ± 0.0161,0.5921 ± 0.0153,0.6070 ± 0.0160,0.8353 ± 0.0088


### MOTOR DOMAIN EXAM


In [45]:
domain_cols = M_cols 
X_HY43_data_M = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_HY4_full.csv", index_col=0)[domain_cols]
y_HY43_data_M = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY4_3_full.csv", index_col=0)
print("X_HY43_data shape:", X_HY43_data_M.shape)
print("y_HY43_data shape:", y_HY43_data_M.shape)
X_HY43_data_M.head()

X_HY43_data shape: (909, 165)
y_HY43_data shape: (909, 1)


,NP3SPCH_mean,NP3SPCH_min,NP3SPCH_max,NP3SPCH_var,NP3SPCH_std,NP3FACXP_mean,NP3FACXP_min,NP3FACXP_max,NP3FACXP_var,NP3FACXP_std,...,NP3RTALJ_mean,NP3RTALJ_min,NP3RTALJ_max,NP3RTALJ_var,NP3RTALJ_std,NP3RTCON_mean,NP3RTCON_min,NP3RTCON_max,NP3RTCON_var,NP3RTCON_std
PATNO,,,,,,,,,,,,,,,,,,,,,
3003,1.666667,1.0,2.0,0.333333,0.57735,1.666667,1.0,2.0,0.333333,0.57735,...,0.0,0.0,0.0,0.0,0.0,0.666667,0.0,1.0,0.333333,0.577350
3018,1.000000,1.0,1.0,0.000000,0.00000,1.666667,1.0,2.0,0.333333,0.57735,...,0.0,0.0,0.0,0.0,0.0,2.666667,1.0,4.0,2.333333,1.527525
3020,0.666667,0.0,1.0,0.333333,0.57735,1.666667,1.0,2.0,0.333333,0.57735,...,0.0,0.0,0.0,0.0,0.0,2.666667,2.0,4.0,1.333333,1.154701
3028,1.333333,1.0,2.0,0.333333,0.57735,2.000000,2.0,2.0,0.000000,0.00000,...,0.0,0.0,0.0,0.0,0.0,1.333333,0.0,3.0,2.333333,1.527525
3051,0.333333,0.0,1.0,0.333333,0.57735,1.000000,1.0,1.0,0.000000,0.00000,...,0.0,0.0,0.0,0.0,0.0,1.333333,1.0,2.0,0.333333,0.577350


In [46]:
df_HY43_M = evaluate_models_10x10_oof_and_test(
    X_df=X_HY43_data_M, 
    y_df=y_HY43_data_M, 
    models=classification_models, 
    random_state=42
)

df_HY43_M.to_csv(full_set_path_HY4 / "HY43_M_StandardScaler.csv")

df_HY43_M_min_max = evaluate_models_10x10_oof_and_test(
    X_df=X_HY43_data_M, 
    y_df=y_HY43_data_M, 
    models=classification_models,
    scaler = MinMaxScaler(), 
    random_state=42
)
df_HY43_M_min_max.to_csv(full_set_path_HY4 / "HY43_M_MinMaxScaler.csv")

Evaluating decision_tree...
Evaluating random_forest...
Evaluating extra_trees...
Evaluating xgboost...
Evaluating adaboost...
Evaluating svm...
Evaluating logistic_regression...
Evaluating knn...
Evaluating gaussian_nb...
Evaluating decision_tree...
Evaluating random_forest...
Evaluating extra_trees...
Evaluating xgboost...
Evaluating adaboost...
Evaluating svm...
Evaluating logistic_regression...
Evaluating knn...
Evaluating gaussian_nb...


In [47]:
df_HY43_M.head(10)

,Accuracy_Testing,Precision_macro_Testing,Recall_macro_Testing,F1_macro_Testing,AUC_macro_Testing,Precision_weighted_Testing,Recall_weighted_Testing,F1_weighted_Testing,AUC_weighted_Testing,Accuracy_CV,Precision_macro_CV,Recall_macro_CV,F1_macro_CV,AUC_macro_CV,Precision_weighted_CV,Recall_weighted_CV,F1_weighted_CV,AUC_weighted_CV
decision_tree,0.7623 ± 0.0190,0.6332 ± 0.0158,0.6335 ± 0.0163,0.6316 ± 0.0154,0.7510 ± 0.0138,0.7717 ± 0.0140,0.7623 ± 0.0190,0.7657 ± 0.0153,0.8155 ± 0.0138,0.7613 ± 0.0140,0.6321 ± 0.0232,0.6324 ± 0.0262,0.6315 ± 0.0246,0.7472 ± 0.0168,0.7723 ± 0.0115,0.7613 ± 0.0140,0.7664 ± 0.0120,0.8139 ± 0.0095
random_forest,0.8619 ± 0.0127,0.7752 ± 0.0488,0.6739 ± 0.0160,0.6691 ± 0.0222,0.8921 ± 0.0168,0.8377 ± 0.0214,0.8619 ± 0.0127,0.8300 ± 0.0144,0.9400 ± 0.0092,0.8593 ± 0.0060,0.7718 ± 0.0284,0.6695 ± 0.0130,0.6636 ± 0.0209,0.8851 ± 0.0092,0.8345 ± 0.0125,0.8593 ± 0.0060,0.8269 ± 0.0093,0.9361 ± 0.0040
extra_trees,0.8582 ± 0.0115,0.7632 ± 0.0620,0.6712 ± 0.0156,0.6661 ± 0.0229,0.8903 ± 0.0136,0.8323 ± 0.0233,0.8582 ± 0.0115,0.8269 ± 0.0129,0.9380 ± 0.0069,0.8566 ± 0.0049,0.7738 ± 0.0193,0.6672 ± 0.0109,0.6613 ± 0.0177,0.8896 ± 0.0084,0.8334 ± 0.0088,0.8566 ± 0.0049,0.8243 ± 0.0077,0.9375 ± 0.0039
xgboost,0.8344 ± 0.0140,0.6829 ± 0.0364,0.6626 ± 0.0160,0.6606 ± 0.0215,0.8512 ± 0.0165,0.8037 ± 0.0160,0.8344 ± 0.0140,0.8143 ± 0.0126,0.9162 ± 0.0103,0.8387 ± 0.0072,0.7011 ± 0.0233,0.6691 ± 0.0140,0.6701 ± 0.0186,0.8507 ± 0.0103,0.8094 ± 0.0107,0.8387 ± 0.0072,0.8185 ± 0.0087,0.9162 ± 0.0059
adaboost,0.8190 ± 0.0127,0.6506 ± 0.0155,0.6485 ± 0.0110,0.6443 ± 0.0135,0.8459 ± 0.0214,0.7929 ± 0.0081,0.8190 ± 0.0127,0.8036 ± 0.0080,0.9167 ± 0.0086,0.8149 ± 0.0100,0.6521 ± 0.0209,0.6479 ± 0.0137,0.6462 ± 0.0163,0.8328 ± 0.0120,0.7906 ± 0.0087,0.8149 ± 0.0100,0.8012 ± 0.0084,0.9110 ± 0.0042
svm,0.8568 ± 0.0080,0.7559 ± 0.1511,0.6492 ± 0.0107,0.6225 ± 0.0179,0.8610 ± 0.0118,0.8250 ± 0.0529,0.8568 ± 0.0080,0.8125 ± 0.0100,0.9249 ± 0.0054,0.8520 ± 0.0055,0.7214 ± 0.1071,0.6437 ± 0.0093,0.6166 ± 0.0166,0.8554 ± 0.0097,0.8113 ± 0.0350,0.8520 ± 0.0055,0.8083 ± 0.0080,0.9194 ± 0.0053
logistic_regression,0.7491 ± 0.0070,0.6339 ± 0.0084,0.6286 ± 0.0116,0.6267 ± 0.0095,0.8112 ± 0.0134,0.7857 ± 0.0048,0.7491 ± 0.0070,0.7643 ± 0.0047,0.8863 ± 0.0068,0.7539 ± 0.0172,0.6347 ± 0.0135,0.6318 ± 0.0185,0.6301 ± 0.0167,0.8090 ± 0.0156,0.7843 ± 0.0085,0.7539 ± 0.0172,0.7673 ± 0.0132,0.8817 ± 0.0088
knn,0.8070 ± 0.0228,0.6784 ± 0.0523,0.6418 ± 0.0281,0.6405 ± 0.0344,0.8463 ± 0.0192,0.7855 ± 0.0258,0.8070 ± 0.0228,0.7851 ± 0.0249,0.9065 ± 0.0149,0.8028 ± 0.0090,0.6636 ± 0.0322,0.6377 ± 0.0180,0.6359 ± 0.0233,0.8410 ± 0.0115,0.7801 ± 0.0145,0.8028 ± 0.0090,0.7819 ± 0.0112,0.9020 ± 0.0060
gaussian_nb,0.7179 ± 0.0461,0.6750 ± 0.0248,0.6635 ± 0.0406,0.6284 ± 0.0419,0.8564 ± 0.0171,0.8297 ± 0.0218,0.7179 ± 0.0461,0.7400 ± 0.0428,0.9154 ± 0.0108,0.6928 ± 0.0354,0.6523 ± 0.0106,0.6327 ± 0.0183,0.6028 ± 0.0259,0.8434 ± 0.0092,0.8114 ± 0.0094,0.6928 ± 0.0354,0.7202 ± 0.0336,0.9054 ± 0.0051


In [48]:
df_HY43_M_min_max.head(10)

,Accuracy_Testing,Precision_macro_Testing,Recall_macro_Testing,F1_macro_Testing,AUC_macro_Testing,Precision_weighted_Testing,Recall_weighted_Testing,F1_weighted_Testing,AUC_weighted_Testing,Accuracy_CV,Precision_macro_CV,Recall_macro_CV,F1_macro_CV,AUC_macro_CV,Precision_weighted_CV,Recall_weighted_CV,F1_weighted_CV,AUC_weighted_CV
decision_tree,0.7623 ± 0.0176,0.6323 ± 0.0164,0.6327 ± 0.0175,0.6309 ± 0.0162,0.7504 ± 0.0146,0.7708 ± 0.0146,0.7623 ± 0.0176,0.7653 ± 0.0149,0.8151 ± 0.0136,0.7607 ± 0.0145,0.6313 ± 0.0242,0.6315 ± 0.0276,0.6306 ± 0.0257,0.7466 ± 0.0176,0.7716 ± 0.0122,0.7607 ± 0.0145,0.7658 ± 0.0126,0.8134 ± 0.0099
random_forest,0.8630 ± 0.0126,0.7774 ± 0.0506,0.6756 ± 0.0173,0.6713 ± 0.0243,0.8919 ± 0.0167,0.8391 ± 0.0222,0.8630 ± 0.0126,0.8314 ± 0.0149,0.9399 ± 0.0091,0.8593 ± 0.0058,0.7721 ± 0.0307,0.6695 ± 0.0122,0.6637 ± 0.0198,0.8848 ± 0.0091,0.8346 ± 0.0126,0.8593 ± 0.0058,0.8269 ± 0.0087,0.9360 ± 0.0041
extra_trees,0.8582 ± 0.0115,0.7632 ± 0.0620,0.6712 ± 0.0156,0.6661 ± 0.0229,0.8903 ± 0.0136,0.8323 ± 0.0233,0.8582 ± 0.0115,0.8269 ± 0.0129,0.9380 ± 0.0069,0.8566 ± 0.0049,0.7738 ± 0.0193,0.6672 ± 0.0109,0.6613 ± 0.0177,0.8896 ± 0.0084,0.8334 ± 0.0088,0.8566 ± 0.0049,0.8243 ± 0.0077,0.9375 ± 0.0039
xgboost,0.8344 ± 0.0140,0.6829 ± 0.0364,0.6626 ± 0.0160,0.6606 ± 0.0215,0.8512 ± 0.0165,0.8037 ± 0.0160,0.8344 ± 0.0140,0.8143 ± 0.0126,0.9162 ± 0.0103,0.8387 ± 0.0072,0.7011 ± 0.0233,0.6691 ± 0.0140,0.6701 ± 0.0186,0.8507 ± 0.0103,0.8094 ± 0.0107,0.8387 ± 0.0072,0.8185 ± 0.0087,0.9162 ± 0.0059
adaboost,0.8190 ± 0.0127,0.6506 ± 0.0155,0.6485 ± 0.0110,0.6443 ± 0.0135,0.8459 ± 0.0214,0.7929 ± 0.0081,0.8190 ± 0.0127,0.8036 ± 0.0080,0.9167 ± 0.0086,0.8149 ± 0.0100,0.6521 ± 0.0209,0.6479 ± 0.0137,0.6462 ± 0.0163,0.8328 ± 0.0120,0.7906 ± 0.0087,0.8149 ± 0.0100,0.8012 ± 0.0084,0.9110 ± 0.0042
svm,0.8568 ± 0.0074,0.6811 ± 0.1133,0.6483 ± 0.0104,0.6204 ± 0.0185,0.8655 ± 0.0142,0.8010 ± 0.0413,0.8568 ± 0.0074,0.8123 ± 0.0104,0.9276 ± 0.0058,0.8525 ± 0.0047,0.7130 ± 0.0921,0.6440 ± 0.0086,0.6170 ± 0.0155,0.8603 ± 0.0103,0.8087 ± 0.0313,0.8525 ± 0.0047,0.8087 ± 0.0070,0.9222 ± 0.0055
logistic_regression,0.7645 ± 0.0167,0.6565 ± 0.0168,0.6535 ± 0.0237,0.6469 ± 0.0181,0.8532 ± 0.0183,0.8098 ± 0.0135,0.7645 ± 0.0167,0.7818 ± 0.0125,0.9183 ± 0.0092,0.7648 ± 0.0168,0.6541 ± 0.0151,0.6529 ± 0.0207,0.6476 ± 0.0184,0.8519 ± 0.0184,0.8045 ± 0.0108,0.7648 ± 0.0168,0.7811 ± 0.0139,0.9157 ± 0.0093
knn,0.8161 ± 0.0148,0.6814 ± 0.0318,0.6520 ± 0.0198,0.6514 ± 0.0248,0.8393 ± 0.0195,0.7937 ± 0.0173,0.8161 ± 0.0148,0.7958 ± 0.0170,0.9040 ± 0.0126,0.8193 ± 0.0084,0.6826 ± 0.0219,0.6520 ± 0.0136,0.6512 ± 0.0180,0.8431 ± 0.0091,0.7940 ± 0.0106,0.8193 ± 0.0084,0.7979 ± 0.0093,0.9040 ± 0.0058
gaussian_nb,0.7278 ± 0.0457,0.6781 ± 0.0278,0.6701 ± 0.0439,0.6373 ± 0.0429,0.8584 ± 0.0171,0.8314 ± 0.0235,0.7278 ± 0.0457,0.7497 ± 0.0413,0.9163 ± 0.0106,0.6995 ± 0.0322,0.6545 ± 0.0111,0.6374 ± 0.0174,0.6092 ± 0.0230,0.8450 ± 0.0092,0.8131 ± 0.0099,0.6995 ± 0.0322,0.7276 ± 0.0298,0.9062 ± 0.0052


## HY4


### FULL DOMAIN


In [49]:
domain_cols = SC_cols + NM_cols + IM_cols + M_cols
X_HY4_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_HY4_full.csv", index_col=0)[domain_cols]
y_HY4_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY4_full.csv", index_col=0)
print("X_HY4_data shape:", X_HY4_data.shape)
print("y_HY4_data shape:", y_HY4_data.shape)
X_HY4_data.head()

X_HY4_data shape: (909, 301)
y_HY4_data shape: (909, 1)


,ENRLLRRK2,ENRLGBA,SEX,RAWHITE,EDUCYRS,AGE_AT_VISIT,NP1COG_mean,NP1COG_min,NP1COG_max,NP1COG_var,...,NP3RTALJ_mean,NP3RTALJ_min,NP3RTALJ_max,NP3RTALJ_var,NP3RTALJ_std,NP3RTCON_mean,NP3RTCON_min,NP3RTCON_max,NP3RTCON_var,NP3RTCON_std
PATNO,,,,,,,,,,,,,,,,,,,,,
3003,0,0,0.0,1.0,16.0,59.7,0.000000,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.666667,0.0,1.0,0.333333,0.577350
3018,0,0,0.0,1.0,16.0,63.6,0.333333,0.0,1.0,0.333333,...,0.0,0.0,0.0,0.0,0.0,2.666667,1.0,4.0,2.333333,1.527525
3020,0,0,0.0,1.0,15.0,77.0,1.000000,1.0,1.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,2.666667,2.0,4.0,1.333333,1.154701
3028,0,0,1.0,1.0,22.0,78.8,0.666667,0.0,1.0,0.333333,...,0.0,0.0,0.0,0.0,0.0,1.333333,0.0,3.0,2.333333,1.527525
3051,0,0,1.0,1.0,18.0,74.7,0.000000,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,1.333333,1.0,2.0,0.333333,0.577350


In [50]:
df_HY4_full = evaluate_models_10x10_oof_and_test(
    X_df=X_HY4_data, 
    y_df=y_HY4_data, 
    models=classification_models, 
    random_state=42
)

df_HY4_full.to_csv(full_set_path_HY4 / "HY4_full_StandardScaler.csv")

df_HY4_full_min_max = evaluate_models_10x10_oof_and_test(
    X_df=X_HY4_data, 
    y_df=y_HY4_data, 
    models=classification_models,
    scaler = MinMaxScaler(), 
    random_state=42
)
df_HY4_full_min_max.to_csv(full_set_path_HY4 / "HY4_full_MinMaxScaler.csv")


Evaluating decision_tree...
Evaluating random_forest...
Evaluating extra_trees...
Evaluating xgboost...
Evaluating adaboost...
Evaluating svm...
Evaluating logistic_regression...
Evaluating knn...
Evaluating gaussian_nb...
Evaluating decision_tree...
Evaluating random_forest...
Evaluating extra_trees...
Evaluating xgboost...
Evaluating adaboost...
Evaluating svm...
Evaluating logistic_regression...
Evaluating knn...
Evaluating gaussian_nb...


In [51]:
df_HY4_full.head(10)

,Accuracy_Testing,Precision_macro_Testing,Recall_macro_Testing,F1_macro_Testing,AUC_macro_Testing,Precision_weighted_Testing,Recall_weighted_Testing,F1_weighted_Testing,AUC_weighted_Testing,Accuracy_CV,Precision_macro_CV,Recall_macro_CV,F1_macro_CV,AUC_macro_CV,Precision_weighted_CV,Recall_weighted_CV,F1_weighted_CV,AUC_weighted_CV
decision_tree,0.7348 ± 0.0242,0.5333 ± 0.0319,0.5357 ± 0.0293,0.5324 ± 0.0305,0.7192 ± 0.0164,0.7452 ± 0.0232,0.7348 ± 0.0242,0.7390 ± 0.0232,0.8053 ± 0.0187,0.7203 ± 0.0137,0.5183 ± 0.0291,0.5258 ± 0.0325,0.5201 ± 0.0303,0.7121 ± 0.0182,0.7362 ± 0.0132,0.7203 ± 0.0137,0.7275 ± 0.0130,0.7968 ± 0.0101
random_forest,0.8443 ± 0.0065,0.6293 ± 0.1292,0.5062 ± 0.0131,0.4922 ± 0.0248,0.8918 ± 0.0132,0.8146 ± 0.0380,0.8443 ± 0.0065,0.7963 ± 0.0080,0.9309 ± 0.0089,0.8338 ± 0.0077,0.6643 ± 0.1037,0.4962 ± 0.0114,0.4772 ± 0.0188,0.8929 ± 0.0089,0.8136 ± 0.0263,0.8338 ± 0.0077,0.7822 ± 0.0097,0.9279 ± 0.0047
extra_trees,0.8458 ± 0.0103,0.7357 ± 0.1126,0.5264 ± 0.0273,0.5263 ± 0.0419,0.8929 ± 0.0132,0.8353 ± 0.0198,0.8458 ± 0.0103,0.8016 ± 0.0122,0.9296 ± 0.0081,0.8324 ± 0.0070,0.7223 ± 0.0969,0.5115 ± 0.0208,0.5055 ± 0.0338,0.8936 ± 0.0084,0.8152 ± 0.0183,0.8324 ± 0.0070,0.7851 ± 0.0098,0.9280 ± 0.0049
xgboost,0.8330 ± 0.0156,0.6975 ± 0.1035,0.5716 ± 0.0471,0.5827 ± 0.0506,0.8800 ± 0.0172,0.8042 ± 0.0231,0.8330 ± 0.0156,0.8033 ± 0.0142,0.9243 ± 0.0080,0.8127 ± 0.0080,0.6167 ± 0.0318,0.5316 ± 0.0147,0.5425 ± 0.0200,0.8700 ± 0.0124,0.7727 ± 0.0117,0.8127 ± 0.0080,0.7827 ± 0.0086,0.9173 ± 0.0062
adaboost,0.8073 ± 0.0196,0.6366 ± 0.0720,0.5731 ± 0.0573,0.5816 ± 0.0540,0.8588 ± 0.0132,0.7790 ± 0.0139,0.8073 ± 0.0196,0.7875 ± 0.0143,0.9062 ± 0.0096,0.7719 ± 0.0131,0.5886 ± 0.0488,0.5364 ± 0.0305,0.5502 ± 0.0347,0.8534 ± 0.0058,0.7533 ± 0.0135,0.7719 ± 0.0131,0.7605 ± 0.0123,0.8984 ± 0.0045
svm,0.8322 ± 0.0096,0.5166 ± 0.1147,0.4842 ± 0.0066,0.4538 ± 0.0094,0.8838 ± 0.0071,0.7658 ± 0.0506,0.8322 ± 0.0096,0.7754 ± 0.0092,0.9208 ± 0.0048,0.8245 ± 0.0036,0.5198 ± 0.0832,0.4808 ± 0.0032,0.4506 ± 0.0066,0.8798 ± 0.0036,0.7620 ± 0.0357,0.8245 ± 0.0036,0.7683 ± 0.0036,0.9164 ± 0.0035
logistic_regression,0.7322 ± 0.0332,0.5369 ± 0.0424,0.5504 ± 0.0507,0.5397 ± 0.0457,0.8131 ± 0.0282,0.7479 ± 0.0224,0.7322 ± 0.0332,0.7379 ± 0.0272,0.8721 ± 0.0153,0.7171 ± 0.0133,0.5260 ± 0.0285,0.5305 ± 0.0294,0.5269 ± 0.0288,0.8016 ± 0.0197,0.7362 ± 0.0117,0.7171 ± 0.0133,0.7259 ± 0.0122,0.8631 ± 0.0127
knn,0.7696 ± 0.0173,0.5800 ± 0.1376,0.4892 ± 0.0432,0.4951 ± 0.0621,0.8080 ± 0.0261,0.7392 ± 0.0310,0.7696 ± 0.0173,0.7396 ± 0.0213,0.8821 ± 0.0138,0.7575 ± 0.0075,0.5921 ± 0.0885,0.4760 ± 0.0127,0.4781 ± 0.0198,0.7975 ± 0.0111,0.7275 ± 0.0140,0.7575 ± 0.0075,0.7277 ± 0.0071,0.8760 ± 0.0065
gaussian_nb,0.6172 ± 0.0413,0.5185 ± 0.0382,0.6190 ± 0.0539,0.4739 ± 0.0464,0.8421 ± 0.0330,0.7639 ± 0.0403,0.6172 ± 0.0413,0.6258 ± 0.0462,0.8917 ± 0.0133,0.6086 ± 0.0260,0.5133 ± 0.0188,0.5998 ± 0.0354,0.4728 ± 0.0246,0.8178 ± 0.0164,0.7550 ± 0.0223,0.6086 ± 0.0260,0.6236 ± 0.0311,0.8797 ± 0.0070


In [52]:
df_HY4_full_min_max.head(10)

,Accuracy_Testing,Precision_macro_Testing,Recall_macro_Testing,F1_macro_Testing,AUC_macro_Testing,Precision_weighted_Testing,Recall_weighted_Testing,F1_weighted_Testing,AUC_weighted_Testing,Accuracy_CV,Precision_macro_CV,Recall_macro_CV,F1_macro_CV,AUC_macro_CV,Precision_weighted_CV,Recall_weighted_CV,F1_weighted_CV,AUC_weighted_CV
decision_tree,0.7348 ± 0.0247,0.5346 ± 0.0305,0.5363 ± 0.0299,0.5334 ± 0.0298,0.7195 ± 0.0169,0.7453 ± 0.0235,0.7348 ± 0.0247,0.7390 ± 0.0236,0.8052 ± 0.0190,0.7206 ± 0.0139,0.5188 ± 0.0288,0.5263 ± 0.0325,0.5205 ± 0.0302,0.7124 ± 0.0182,0.7365 ± 0.0124,0.7206 ± 0.0139,0.7278 ± 0.0127,0.7971 ± 0.0098
random_forest,0.8443 ± 0.0076,0.6313 ± 0.1287,0.5056 ± 0.0132,0.4911 ± 0.0247,0.8939 ± 0.0118,0.8152 ± 0.0389,0.8443 ± 0.0076,0.7958 ± 0.0094,0.9313 ± 0.0088,0.8336 ± 0.0074,0.6539 ± 0.1146,0.4963 ± 0.0117,0.4777 ± 0.0194,0.8922 ± 0.0090,0.8090 ± 0.0288,0.8336 ± 0.0074,0.7822 ± 0.0094,0.9277 ± 0.0047
extra_trees,0.8458 ± 0.0103,0.7357 ± 0.1126,0.5264 ± 0.0273,0.5263 ± 0.0419,0.8929 ± 0.0132,0.8353 ± 0.0198,0.8458 ± 0.0103,0.8016 ± 0.0122,0.9296 ± 0.0081,0.8324 ± 0.0070,0.7223 ± 0.0969,0.5115 ± 0.0208,0.5055 ± 0.0338,0.8936 ± 0.0084,0.8152 ± 0.0183,0.8324 ± 0.0070,0.7851 ± 0.0098,0.9280 ± 0.0049
xgboost,0.8326 ± 0.0154,0.6952 ± 0.0998,0.5707 ± 0.0478,0.5812 ± 0.0518,0.8810 ± 0.0171,0.8030 ± 0.0215,0.8326 ± 0.0154,0.8025 ± 0.0147,0.9246 ± 0.0079,0.8127 ± 0.0080,0.6170 ± 0.0318,0.5316 ± 0.0147,0.5425 ± 0.0200,0.8701 ± 0.0124,0.7728 ± 0.0117,0.8127 ± 0.0080,0.7827 ± 0.0086,0.9174 ± 0.0062
adaboost,0.8077 ± 0.0185,0.6368 ± 0.0717,0.5734 ± 0.0571,0.5818 ± 0.0538,0.8588 ± 0.0132,0.7792 ± 0.0137,0.8077 ± 0.0185,0.7878 ± 0.0137,0.9062 ± 0.0096,0.7719 ± 0.0131,0.5886 ± 0.0488,0.5364 ± 0.0305,0.5502 ± 0.0347,0.8534 ± 0.0058,0.7533 ± 0.0135,0.7719 ± 0.0131,0.7605 ± 0.0123,0.8984 ± 0.0045
svm,0.8322 ± 0.0084,0.4918 ± 0.1069,0.4835 ± 0.0067,0.4524 ± 0.0099,0.8871 ± 0.0074,0.7551 ± 0.0490,0.8322 ± 0.0084,0.7751 ± 0.0086,0.9241 ± 0.0052,0.8242 ± 0.0048,0.4777 ± 0.0570,0.4796 ± 0.0035,0.4486 ± 0.0064,0.8851 ± 0.0041,0.7436 ± 0.0244,0.8242 ± 0.0048,0.7677 ± 0.0046,0.9200 ± 0.0039
logistic_regression,0.7399 ± 0.0276,0.5558 ± 0.0414,0.5772 ± 0.0527,0.5600 ± 0.0460,0.8572 ± 0.0149,0.7690 ± 0.0190,0.7399 ± 0.0276,0.7506 ± 0.0220,0.9029 ± 0.0116,0.7316 ± 0.0068,0.5441 ± 0.0154,0.5562 ± 0.0221,0.5467 ± 0.0173,0.8528 ± 0.0114,0.7584 ± 0.0091,0.7316 ± 0.0068,0.7428 ± 0.0070,0.8970 ± 0.0079
knn,0.7835 ± 0.0119,0.5798 ± 0.1165,0.5074 ± 0.0377,0.5149 ± 0.0561,0.8110 ± 0.0248,0.7498 ± 0.0216,0.7835 ± 0.0119,0.7563 ± 0.0139,0.8937 ± 0.0123,0.7753 ± 0.0101,0.6211 ± 0.0923,0.4919 ± 0.0155,0.4977 ± 0.0241,0.7997 ± 0.0133,0.7440 ± 0.0161,0.7753 ± 0.0101,0.7466 ± 0.0108,0.8887 ± 0.0069
gaussian_nb,0.6388 ± 0.0325,0.5293 ± 0.0331,0.6321 ± 0.0501,0.4976 ± 0.0375,0.8429 ± 0.0332,0.7718 ± 0.0331,0.6388 ± 0.0325,0.6527 ± 0.0337,0.8920 ± 0.0132,0.6242 ± 0.0269,0.5182 ± 0.0194,0.6061 ± 0.0373,0.4886 ± 0.0266,0.8189 ± 0.0168,0.7573 ± 0.0206,0.6242 ± 0.0269,0.6429 ± 0.0303,0.8803 ± 0.0074


### NON MOTOR DOMAIN


In [53]:
domain_cols = NM_cols 
X_HY4_data_NM = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_HY4_full.csv", index_col=0)[domain_cols]
y_HY4_data_NM = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY4_full.csv", index_col=0)
print("X_HY4_data_NM shape:", X_HY4_data_NM.shape)
print("y_HY4_data_NM shape:", y_HY4_data_NM.shape)
X_HY4_data_NM.head()

X_HY4_data_NM shape: (909, 65)
y_HY4_data_NM shape: (909, 1)


,NP1COG_mean,NP1COG_min,NP1COG_max,NP1COG_var,NP1COG_std,NP1HALL_mean,NP1HALL_min,NP1HALL_max,NP1HALL_var,NP1HALL_std,...,NP1LTHD_mean,NP1LTHD_min,NP1LTHD_max,NP1LTHD_var,NP1LTHD_std,NP1FATG_mean,NP1FATG_min,NP1FATG_max,NP1FATG_var,NP1FATG_std
PATNO,,,,,,,,,,,,,,,,,,,,,
3003,0.000000,0.0,0.0,0.000000,0.00000,0.000000,0.0,0.0,0.000000,0.00000,...,1.333333,1.0,2.0,0.333333,0.57735,1.000000,1.0,1.0,0.000000,0.00000
3018,0.333333,0.0,1.0,0.333333,0.57735,0.000000,0.0,0.0,0.000000,0.00000,...,0.333333,0.0,1.0,0.333333,0.57735,1.000000,0.0,2.0,1.000000,1.00000
3020,1.000000,1.0,1.0,0.000000,0.00000,0.333333,0.0,1.0,0.333333,0.57735,...,0.333333,0.0,1.0,0.333333,0.57735,1.333333,1.0,2.0,0.333333,0.57735
3028,0.666667,0.0,1.0,0.333333,0.57735,0.000000,0.0,0.0,0.000000,0.00000,...,0.000000,0.0,0.0,0.000000,0.00000,0.666667,0.0,1.0,0.333333,0.57735
3051,0.000000,0.0,0.0,0.000000,0.00000,0.000000,0.0,0.0,0.000000,0.00000,...,0.000000,0.0,0.0,0.000000,0.00000,1.000000,1.0,1.0,0.000000,0.00000


In [54]:
df_HY4_NM = evaluate_models_10x10_oof_and_test(
    X_df=X_HY4_data_NM, 
    y_df=y_HY4_data_NM, 
    models=classification_models, 
    random_state=42
)

df_HY4_NM.to_csv(full_set_path_HY4 / "HY4_NM_StandardScaler.csv")

df_HY4_NM_min_max = evaluate_models_10x10_oof_and_test(
    X_df=X_HY4_data_NM, 
    y_df=y_HY4_data_NM, 
    models=classification_models,
    scaler = MinMaxScaler(), 
    random_state=42
)
df_HY4_NM_min_max.to_csv(full_set_path_HY4 / "HY4_NM_MinMaxScaler.csv")


Evaluating decision_tree...
Evaluating random_forest...
Evaluating extra_trees...
Evaluating xgboost...
Evaluating adaboost...
Evaluating svm...
Evaluating logistic_regression...
Evaluating knn...
Evaluating gaussian_nb...
Evaluating decision_tree...
Evaluating random_forest...
Evaluating extra_trees...
Evaluating xgboost...
Evaluating adaboost...
Evaluating svm...
Evaluating logistic_regression...
Evaluating knn...
Evaluating gaussian_nb...


In [55]:
df_HY4_NM.head(10)

,Accuracy_Testing,Precision_macro_Testing,Recall_macro_Testing,F1_macro_Testing,AUC_macro_Testing,Precision_weighted_Testing,Recall_weighted_Testing,F1_weighted_Testing,AUC_weighted_Testing,Accuracy_CV,Precision_macro_CV,Recall_macro_CV,F1_macro_CV,AUC_macro_CV,Precision_weighted_CV,Recall_weighted_CV,F1_weighted_CV,AUC_weighted_CV
decision_tree,0.4088 ± 0.0337,0.2743 ± 0.0299,0.2739 ± 0.0314,0.2729 ± 0.0307,0.5129 ± 0.0207,0.4144 ± 0.0289,0.4088 ± 0.0337,0.4107 ± 0.0317,0.5148 ± 0.0260,0.4035 ± 0.0181,0.2696 ± 0.0175,0.2718 ± 0.0192,0.2700 ± 0.0180,0.5122 ± 0.0123,0.4115 ± 0.0190,0.4035 ± 0.0181,0.4071 ± 0.0184,0.5154 ± 0.0148
random_forest,0.4835 ± 0.0250,0.2450 ± 0.0126,0.2793 ± 0.0145,0.2609 ± 0.0134,0.6160 ± 0.0295,0.4246 ± 0.0218,0.4835 ± 0.0250,0.4520 ± 0.0232,0.5935 ± 0.0240,0.4800 ± 0.0194,0.2698 ± 0.0343,0.2806 ± 0.0125,0.2652 ± 0.0135,0.5934 ± 0.0246,0.4322 ± 0.0238,0.4800 ± 0.0194,0.4502 ± 0.0184,0.5737 ± 0.0191
extra_trees,0.4839 ± 0.0214,0.2624 ± 0.0286,0.2815 ± 0.0116,0.2654 ± 0.0105,0.6007 ± 0.0249,0.4330 ± 0.0200,0.4839 ± 0.0214,0.4540 ± 0.0189,0.5839 ± 0.0196,0.4728 ± 0.0206,0.2809 ± 0.0300,0.2793 ± 0.0127,0.2676 ± 0.0132,0.5854 ± 0.0178,0.4334 ± 0.0209,0.4728 ± 0.0206,0.4463 ± 0.0188,0.5678 ± 0.0210
xgboost,0.4861 ± 0.0154,0.2758 ± 0.0208,0.2891 ± 0.0140,0.2798 ± 0.0164,0.5891 ± 0.0201,0.4482 ± 0.0125,0.4861 ± 0.0154,0.4651 ± 0.0140,0.5785 ± 0.0197,0.4557 ± 0.0166,0.2675 ± 0.0220,0.2725 ± 0.0135,0.2657 ± 0.0160,0.5779 ± 0.0171,0.4239 ± 0.0182,0.4557 ± 0.0166,0.4371 ± 0.0170,0.5606 ± 0.0200
adaboost,0.5033 ± 0.0273,0.2860 ± 0.0860,0.2981 ± 0.0238,0.2811 ± 0.0304,0.6151 ± 0.0240,0.4450 ± 0.0283,0.5033 ± 0.0273,0.4689 ± 0.0253,0.5764 ± 0.0295,0.4881 ± 0.0177,0.2788 ± 0.0460,0.2866 ± 0.0135,0.2706 ± 0.0160,0.5862 ± 0.0194,0.4348 ± 0.0233,0.4881 ± 0.0177,0.4557 ± 0.0165,0.5542 ± 0.0128
svm,0.5209 ± 0.0230,0.2591 ± 0.0122,0.2992 ± 0.0137,0.2762 ± 0.0135,0.6342 ± 0.0239,0.4484 ± 0.0209,0.5209 ± 0.0230,0.4795 ± 0.0230,0.6150 ± 0.0178,0.5046 ± 0.0110,0.2512 ± 0.0056,0.2905 ± 0.0064,0.2682 ± 0.0059,0.6004 ± 0.0148,0.4341 ± 0.0096,0.5046 ± 0.0110,0.4646 ± 0.0102,0.5895 ± 0.0146
logistic_regression,0.3601 ± 0.0360,0.2915 ± 0.0215,0.2785 ± 0.0411,0.2645 ± 0.0281,0.5410 ± 0.0331,0.4589 ± 0.0271,0.3601 ± 0.0360,0.3938 ± 0.0319,0.5691 ± 0.0238,0.3593 ± 0.0240,0.2885 ± 0.0156,0.2867 ± 0.0340,0.2678 ± 0.0220,0.5395 ± 0.0170,0.4478 ± 0.0152,0.3593 ± 0.0240,0.3905 ± 0.0203,0.5685 ± 0.0080
knn,0.4659 ± 0.0224,0.2683 ± 0.0401,0.2735 ± 0.0186,0.2595 ± 0.0222,0.5343 ± 0.0156,0.4230 ± 0.0309,0.4659 ± 0.0224,0.4322 ± 0.0220,0.5556 ± 0.0273,0.4467 ± 0.0164,0.2472 ± 0.0185,0.2612 ± 0.0109,0.2465 ± 0.0119,0.5299 ± 0.0174,0.4020 ± 0.0169,0.4467 ± 0.0164,0.4148 ± 0.0149,0.5382 ± 0.0219
gaussian_nb,0.4143 ± 0.0142,0.2849 ± 0.0465,0.3542 ± 0.0388,0.2145 ± 0.0202,0.6118 ± 0.0137,0.4368 ± 0.0667,0.4143 ± 0.0142,0.3156 ± 0.0152,0.5783 ± 0.0192,0.3874 ± 0.0230,0.2715 ± 0.0338,0.3501 ± 0.0363,0.2113 ± 0.0069,0.5881 ± 0.0236,0.4065 ± 0.0395,0.3874 ± 0.0230,0.3044 ± 0.0062,0.5675 ± 0.0117


In [56]:
df_HY4_NM_min_max.head(10)

,Accuracy_Testing,Precision_macro_Testing,Recall_macro_Testing,F1_macro_Testing,AUC_macro_Testing,Precision_weighted_Testing,Recall_weighted_Testing,F1_weighted_Testing,AUC_weighted_Testing,Accuracy_CV,Precision_macro_CV,Recall_macro_CV,F1_macro_CV,AUC_macro_CV,Precision_weighted_CV,Recall_weighted_CV,F1_weighted_CV,AUC_weighted_CV
decision_tree,0.4095 ± 0.0338,0.2762 ± 0.0330,0.2750 ± 0.0333,0.2743 ± 0.0330,0.5135 ± 0.0216,0.4145 ± 0.0285,0.4095 ± 0.0338,0.4112 ± 0.0316,0.5148 ± 0.0257,0.4030 ± 0.0170,0.2696 ± 0.0177,0.2717 ± 0.0196,0.2701 ± 0.0184,0.5119 ± 0.0125,0.4107 ± 0.0174,0.4030 ± 0.0170,0.4065 ± 0.0170,0.5146 ± 0.0138
random_forest,0.4864 ± 0.0266,0.2463 ± 0.0130,0.2809 ± 0.0153,0.2624 ± 0.0140,0.6170 ± 0.0262,0.4268 ± 0.0225,0.4864 ± 0.0266,0.4545 ± 0.0244,0.5942 ± 0.0233,0.4796 ± 0.0191,0.2682 ± 0.0376,0.2803 ± 0.0123,0.2649 ± 0.0134,0.5936 ± 0.0249,0.4314 ± 0.0236,0.4796 ± 0.0191,0.4499 ± 0.0182,0.5733 ± 0.0201
extra_trees,0.4839 ± 0.0214,0.2624 ± 0.0286,0.2815 ± 0.0116,0.2654 ± 0.0105,0.6007 ± 0.0249,0.4330 ± 0.0200,0.4839 ± 0.0214,0.4540 ± 0.0189,0.5839 ± 0.0196,0.4728 ± 0.0206,0.2809 ± 0.0300,0.2793 ± 0.0127,0.2676 ± 0.0132,0.5854 ± 0.0178,0.4334 ± 0.0209,0.4728 ± 0.0206,0.4463 ± 0.0188,0.5678 ± 0.0210
xgboost,0.4861 ± 0.0154,0.2758 ± 0.0208,0.2891 ± 0.0140,0.2798 ± 0.0164,0.5891 ± 0.0201,0.4482 ± 0.0125,0.4861 ± 0.0154,0.4651 ± 0.0140,0.5785 ± 0.0197,0.4555 ± 0.0170,0.2672 ± 0.0222,0.2724 ± 0.0137,0.2656 ± 0.0162,0.5778 ± 0.0173,0.4237 ± 0.0185,0.4555 ± 0.0170,0.4370 ± 0.0173,0.5606 ± 0.0199
adaboost,0.5033 ± 0.0273,0.2860 ± 0.0860,0.2981 ± 0.0238,0.2811 ± 0.0304,0.6151 ± 0.0240,0.4450 ± 0.0283,0.5033 ± 0.0273,0.4689 ± 0.0253,0.5764 ± 0.0295,0.4881 ± 0.0177,0.2788 ± 0.0460,0.2866 ± 0.0135,0.2706 ± 0.0160,0.5862 ± 0.0194,0.4348 ± 0.0233,0.4881 ± 0.0177,0.4557 ± 0.0165,0.5542 ± 0.0128
svm,0.5209 ± 0.0233,0.2594 ± 0.0123,0.2991 ± 0.0137,0.2760 ± 0.0134,0.6222 ± 0.0232,0.4488 ± 0.0210,0.5209 ± 0.0233,0.4791 ± 0.0228,0.6094 ± 0.0164,0.5035 ± 0.0127,0.2507 ± 0.0066,0.2897 ± 0.0072,0.2671 ± 0.0066,0.5983 ± 0.0181,0.4333 ± 0.0112,0.5035 ± 0.0127,0.4628 ± 0.0114,0.5898 ± 0.0138
logistic_regression,0.3670 ± 0.0383,0.3133 ± 0.0247,0.3259 ± 0.0386,0.2808 ± 0.0235,0.5906 ± 0.0213,0.4826 ± 0.0397,0.3670 ± 0.0383,0.3997 ± 0.0371,0.5850 ± 0.0152,0.3443 ± 0.0214,0.2951 ± 0.0098,0.3097 ± 0.0195,0.2670 ± 0.0139,0.5745 ± 0.0148,0.4524 ± 0.0128,0.3443 ± 0.0214,0.3764 ± 0.0174,0.5713 ± 0.0082
knn,0.4689 ± 0.0297,0.2890 ± 0.0476,0.2774 ± 0.0204,0.2657 ± 0.0240,0.5389 ± 0.0263,0.4331 ± 0.0384,0.4689 ± 0.0297,0.4365 ± 0.0301,0.5570 ± 0.0314,0.4519 ± 0.0145,0.2700 ± 0.0114,0.2688 ± 0.0075,0.2569 ± 0.0080,0.5340 ± 0.0142,0.4136 ± 0.0119,0.4519 ± 0.0145,0.4204 ± 0.0137,0.5352 ± 0.0175
gaussian_nb,0.4172 ± 0.0138,0.2827 ± 0.0342,0.3527 ± 0.0335,0.2197 ± 0.0182,0.6112 ± 0.0132,0.4347 ± 0.0485,0.4172 ± 0.0138,0.3247 ± 0.0138,0.5775 ± 0.0191,0.3899 ± 0.0238,0.2762 ± 0.0322,0.3515 ± 0.0370,0.2175 ± 0.0098,0.5880 ± 0.0235,0.4107 ± 0.0372,0.3899 ± 0.0238,0.3120 ± 0.0087,0.5674 ± 0.0115


### IMPACT MOTOR DOMAIN


In [57]:
domain_cols = IM_cols
X_HY4_data_IM = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_HY4_full.csv", index_col=0)[domain_cols]
y_HY4_data_IM = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY4_full.csv", index_col=0)
print("X_HY4_data_IM shape:", X_HY4_data_IM.shape)
print("y_HY4_data_IM shape:", y_HY4_data_IM.shape)
X_HY4_data_IM.head()

X_HY4_data_IM shape: (909, 65)
y_HY4_data_IM shape: (909, 1)


,NP2SPCH_mean,NP2SPCH_min,NP2SPCH_max,NP2SPCH_var,NP2SPCH_std,NP2SALV_mean,NP2SALV_min,NP2SALV_max,NP2SALV_var,NP2SALV_std,...,NP2WALK_mean,NP2WALK_min,NP2WALK_max,NP2WALK_var,NP2WALK_std,NP2FREZ_mean,NP2FREZ_min,NP2FREZ_max,NP2FREZ_var,NP2FREZ_std
PATNO,,,,,,,,,,,,,,,,,,,,,
3003,0.666667,0.0,2.0,1.333333,1.154701,0.000000,0.0,0.0,0.000000,0.00000,...,1.000000,1.0,1.0,0.000000,0.00000,0.000000,0.0,0.0,0.000000,0.00000
3018,0.333333,0.0,1.0,0.333333,0.577350,2.000000,2.0,2.0,0.000000,0.00000,...,1.000000,1.0,1.0,0.000000,0.00000,0.000000,0.0,0.0,0.000000,0.00000
3020,0.333333,0.0,1.0,0.333333,0.577350,3.000000,3.0,3.0,0.000000,0.00000,...,0.666667,0.0,1.0,0.333333,0.57735,0.333333,0.0,1.0,0.333333,0.57735
3028,1.000000,1.0,1.0,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.00000,...,0.000000,0.0,0.0,0.000000,0.00000,0.000000,0.0,0.0,0.000000,0.00000
3051,1.000000,1.0,1.0,0.000000,0.000000,0.666667,0.0,1.0,0.333333,0.57735,...,0.000000,0.0,0.0,0.000000,0.00000,0.000000,0.0,0.0,0.000000,0.00000


In [58]:
df_HY4_IM = evaluate_models_10x10_oof_and_test(
    X_df=X_HY4_data_IM, 
    y_df=y_HY4_data_IM, 
    models=classification_models, 
    random_state=42
)

df_HY4_IM.to_csv(full_set_path_HY4 / "HY4_IM_StandardScaler.csv")

df_HY4_IM_min_max = evaluate_models_10x10_oof_and_test(
    X_df=X_HY4_data_IM, 
    y_df=y_HY4_data_IM, 
    models=classification_models,
    scaler = MinMaxScaler(), 
    random_state=42
)
df_HY4_IM_min_max.to_csv(full_set_path_HY4 / "HY4_IM_MinMaxScaler.csv")


Evaluating decision_tree...
Evaluating random_forest...
Evaluating extra_trees...
Evaluating xgboost...
Evaluating adaboost...
Evaluating svm...
Evaluating logistic_regression...
Evaluating knn...
Evaluating gaussian_nb...
Evaluating decision_tree...
Evaluating random_forest...
Evaluating extra_trees...
Evaluating xgboost...
Evaluating adaboost...
Evaluating svm...
Evaluating logistic_regression...
Evaluating knn...
Evaluating gaussian_nb...


In [59]:
df_HY43_IM.head(10)

,Accuracy_Testing,Precision_macro_Testing,Recall_macro_Testing,F1_macro_Testing,AUC_macro_Testing,Precision_weighted_Testing,Recall_weighted_Testing,F1_weighted_Testing,AUC_weighted_Testing,Accuracy_CV,Precision_macro_CV,Recall_macro_CV,F1_macro_CV,AUC_macro_CV,Precision_weighted_CV,Recall_weighted_CV,F1_weighted_CV,AUC_weighted_CV
decision_tree,0.6996 ± 0.0319,0.5550 ± 0.0350,0.5556 ± 0.0341,0.5535 ± 0.0339,0.6655 ± 0.0269,0.7007 ± 0.0272,0.6996 ± 0.0319,0.6985 ± 0.0279,0.7282 ± 0.0275,0.6984 ± 0.0184,0.5511 ± 0.0215,0.5526 ± 0.0201,0.5514 ± 0.0209,0.6611 ± 0.0183,0.6945 ± 0.0187,0.6984 ± 0.0184,0.6960 ± 0.0184,0.7250 ± 0.0196
random_forest,0.7835 ± 0.0226,0.5946 ± 0.0519,0.5994 ± 0.0249,0.5854 ± 0.0301,0.7799 ± 0.0162,0.7401 ± 0.0278,0.7835 ± 0.0226,0.7564 ± 0.0232,0.8636 ± 0.0140,0.7675 ± 0.0135,0.5628 ± 0.0207,0.5820 ± 0.0118,0.5648 ± 0.0132,0.7692 ± 0.0116,0.7193 ± 0.0135,0.7675 ± 0.0135,0.7393 ± 0.0121,0.8514 ± 0.0091
extra_trees,0.7667 ± 0.0150,0.5603 ± 0.0210,0.5826 ± 0.0123,0.5659 ± 0.0149,0.7762 ± 0.0195,0.7190 ± 0.0166,0.7667 ± 0.0150,0.7396 ± 0.0149,0.8495 ± 0.0173,0.7591 ± 0.0149,0.5601 ± 0.0313,0.5758 ± 0.0134,0.5590 ± 0.0152,0.7718 ± 0.0117,0.7117 ± 0.0155,0.7591 ± 0.0149,0.7311 ± 0.0136,0.8424 ± 0.0113
xgboost,0.7714 ± 0.0216,0.5941 ± 0.0294,0.5978 ± 0.0179,0.5888 ± 0.0200,0.7688 ± 0.0181,0.7350 ± 0.0163,0.7714 ± 0.0216,0.7502 ± 0.0170,0.8542 ± 0.0147,0.7552 ± 0.0086,0.5817 ± 0.0172,0.5847 ± 0.0109,0.5768 ± 0.0138,0.7535 ± 0.0141,0.7205 ± 0.0113,0.7552 ± 0.0086,0.7352 ± 0.0101,0.8366 ± 0.0110
adaboost,0.8084 ± 0.0221,0.5974 ± 0.1092,0.6080 ± 0.0181,0.5784 ± 0.0194,0.8125 ± 0.0127,0.7447 ± 0.0418,0.8084 ± 0.0221,0.7662 ± 0.0205,0.8837 ± 0.0112,0.7962 ± 0.0084,0.5846 ± 0.0674,0.5988 ± 0.0093,0.5713 ± 0.0134,0.7956 ± 0.0090,0.7339 ± 0.0250,0.7962 ± 0.0084,0.7561 ± 0.0093,0.8686 ± 0.0049
svm,0.8073 ± 0.0192,0.5392 ± 0.0136,0.6047 ± 0.0144,0.5697 ± 0.0137,0.7986 ± 0.0189,0.7201 ± 0.0182,0.8073 ± 0.0192,0.7607 ± 0.0184,0.8715 ± 0.0166,0.7925 ± 0.0095,0.5293 ± 0.0062,0.5926 ± 0.0071,0.5589 ± 0.0066,0.7834 ± 0.0103,0.7081 ± 0.0083,0.7925 ± 0.0095,0.7475 ± 0.0089,0.8577 ± 0.0065
logistic_regression,0.6901 ± 0.0235,0.6094 ± 0.0207,0.5985 ± 0.0272,0.5892 ± 0.0223,0.7986 ± 0.0194,0.7571 ± 0.0201,0.6901 ± 0.0235,0.7126 ± 0.0213,0.8597 ± 0.0171,0.6814 ± 0.0132,0.5991 ± 0.0101,0.5929 ± 0.0155,0.5829 ± 0.0129,0.7855 ± 0.0144,0.7425 ± 0.0081,0.6814 ± 0.0132,0.7030 ± 0.0110,0.8469 ± 0.0101
knn,0.7484 ± 0.0209,0.5610 ± 0.0604,0.5694 ± 0.0211,0.5510 ± 0.0267,0.7584 ± 0.0154,0.7025 ± 0.0269,0.7484 ± 0.0209,0.7168 ± 0.0196,0.8382 ± 0.0125,0.7332 ± 0.0094,0.5504 ± 0.0313,0.5583 ± 0.0100,0.5422 ± 0.0143,0.7428 ± 0.0172,0.6900 ± 0.0128,0.7332 ± 0.0094,0.7040 ± 0.0083,0.8247 ± 0.0131
gaussian_nb,0.6007 ± 0.0228,0.5776 ± 0.0261,0.5371 ± 0.0324,0.5124 ± 0.0270,0.7866 ± 0.0209,0.7234 ± 0.0288,0.6007 ± 0.0228,0.6170 ± 0.0268,0.8518 ± 0.0192,0.5895 ± 0.0152,0.5698 ± 0.0156,0.5388 ± 0.0194,0.5067 ± 0.0167,0.7728 ± 0.0115,0.7108 ± 0.0171,0.5895 ± 0.0152,0.6052 ± 0.0167,0.8351 ± 0.0089


In [60]:
df_HY43_IM_min_max.head(10)

,Accuracy_Testing,Precision_macro_Testing,Recall_macro_Testing,F1_macro_Testing,AUC_macro_Testing,Precision_weighted_Testing,Recall_weighted_Testing,F1_weighted_Testing,AUC_weighted_Testing,Accuracy_CV,Precision_macro_CV,Recall_macro_CV,F1_macro_CV,AUC_macro_CV,Precision_weighted_CV,Recall_weighted_CV,F1_weighted_CV,AUC_weighted_CV
decision_tree,0.7000 ± 0.0324,0.5551 ± 0.0351,0.5559 ± 0.0338,0.5536 ± 0.0339,0.6649 ± 0.0276,0.7002 ± 0.0271,0.7000 ± 0.0324,0.6983 ± 0.0280,0.7273 ± 0.0280,0.6981 ± 0.0193,0.5506 ± 0.0214,0.5524 ± 0.0203,0.5511 ± 0.0209,0.6607 ± 0.0189,0.6937 ± 0.0188,0.6981 ± 0.0193,0.6955 ± 0.0189,0.7245 ± 0.0204
random_forest,0.7835 ± 0.0233,0.5944 ± 0.0519,0.5994 ± 0.0256,0.5855 ± 0.0305,0.7796 ± 0.0164,0.7400 ± 0.0278,0.7835 ± 0.0233,0.7565 ± 0.0237,0.8634 ± 0.0139,0.7653 ± 0.0138,0.5590 ± 0.0201,0.5800 ± 0.0118,0.5627 ± 0.0132,0.7687 ± 0.0118,0.7170 ± 0.0134,0.7653 ± 0.0138,0.7374 ± 0.0122,0.8511 ± 0.0093
extra_trees,0.7667 ± 0.0150,0.5603 ± 0.0210,0.5826 ± 0.0123,0.5659 ± 0.0149,0.7762 ± 0.0195,0.7190 ± 0.0166,0.7667 ± 0.0150,0.7396 ± 0.0149,0.8495 ± 0.0173,0.7591 ± 0.0149,0.5601 ± 0.0313,0.5758 ± 0.0134,0.5590 ± 0.0152,0.7718 ± 0.0117,0.7117 ± 0.0155,0.7591 ± 0.0149,0.7311 ± 0.0136,0.8424 ± 0.0113
xgboost,0.7714 ± 0.0216,0.5941 ± 0.0294,0.5978 ± 0.0179,0.5888 ± 0.0200,0.7688 ± 0.0181,0.7350 ± 0.0163,0.7714 ± 0.0216,0.7502 ± 0.0170,0.8542 ± 0.0147,0.7552 ± 0.0086,0.5817 ± 0.0172,0.5847 ± 0.0109,0.5768 ± 0.0138,0.7535 ± 0.0141,0.7205 ± 0.0113,0.7552 ± 0.0086,0.7352 ± 0.0101,0.8366 ± 0.0110
adaboost,0.8084 ± 0.0221,0.5974 ± 0.1092,0.6080 ± 0.0181,0.5784 ± 0.0194,0.8124 ± 0.0127,0.7447 ± 0.0418,0.8084 ± 0.0221,0.7662 ± 0.0205,0.8837 ± 0.0113,0.7962 ± 0.0084,0.5846 ± 0.0674,0.5988 ± 0.0093,0.5713 ± 0.0134,0.7956 ± 0.0090,0.7339 ± 0.0250,0.7962 ± 0.0084,0.7561 ± 0.0093,0.8686 ± 0.0049
svm,0.8114 ± 0.0183,0.5416 ± 0.0131,0.6077 ± 0.0137,0.5725 ± 0.0131,0.7985 ± 0.0173,0.7232 ± 0.0175,0.8114 ± 0.0183,0.7644 ± 0.0175,0.8734 ± 0.0155,0.7980 ± 0.0087,0.5325 ± 0.0056,0.5967 ± 0.0065,0.5626 ± 0.0061,0.7870 ± 0.0081,0.7123 ± 0.0074,0.7980 ± 0.0087,0.7524 ± 0.0082,0.8609 ± 0.0055
logistic_regression,0.6967 ± 0.0250,0.6097 ± 0.0193,0.5976 ± 0.0296,0.5900 ± 0.0254,0.8129 ± 0.0164,0.7586 ± 0.0162,0.6967 ± 0.0250,0.7165 ± 0.0213,0.8794 ± 0.0132,0.6903 ± 0.0116,0.6073 ± 0.0105,0.5983 ± 0.0156,0.5880 ± 0.0125,0.8015 ± 0.0114,0.7547 ± 0.0090,0.6903 ± 0.0116,0.7113 ± 0.0101,0.8661 ± 0.0088
knn,0.7476 ± 0.0214,0.5713 ± 0.0512,0.5714 ± 0.0219,0.5560 ± 0.0263,0.7675 ± 0.0169,0.7064 ± 0.0256,0.7476 ± 0.0214,0.7181 ± 0.0211,0.8455 ± 0.0150,0.7403 ± 0.0088,0.5618 ± 0.0251,0.5658 ± 0.0104,0.5508 ± 0.0158,0.7480 ± 0.0206,0.6979 ± 0.0126,0.7403 ± 0.0088,0.7111 ± 0.0105,0.8307 ± 0.0160
gaussian_nb,0.6044 ± 0.0216,0.5761 ± 0.0247,0.5374 ± 0.0315,0.5142 ± 0.0259,0.7862 ± 0.0208,0.7212 ± 0.0272,0.6044 ± 0.0216,0.6198 ± 0.0252,0.8512 ± 0.0194,0.5921 ± 0.0153,0.5668 ± 0.0144,0.5357 ± 0.0176,0.5065 ± 0.0154,0.7731 ± 0.0114,0.7077 ± 0.0161,0.5921 ± 0.0153,0.6070 ± 0.0160,0.8353 ± 0.0088


### MOTOR DOMAIN EXAM


In [61]:
domain_cols = M_cols
X_HY4_data_M = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_HY4_full.csv", index_col=0)[domain_cols]
y_HY4_data_M = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY4_full.csv", index_col=0)
print("X_HY4_data_M shape:", X_HY4_data_M.shape)
print("y_HY4_data_M shape:", y_HY4_data_M.shape)
X_HY4_data_M.head()

X_HY4_data_M shape: (909, 165)
y_HY4_data_M shape: (909, 1)


,NP3SPCH_mean,NP3SPCH_min,NP3SPCH_max,NP3SPCH_var,NP3SPCH_std,NP3FACXP_mean,NP3FACXP_min,NP3FACXP_max,NP3FACXP_var,NP3FACXP_std,...,NP3RTALJ_mean,NP3RTALJ_min,NP3RTALJ_max,NP3RTALJ_var,NP3RTALJ_std,NP3RTCON_mean,NP3RTCON_min,NP3RTCON_max,NP3RTCON_var,NP3RTCON_std
PATNO,,,,,,,,,,,,,,,,,,,,,
3003,1.666667,1.0,2.0,0.333333,0.57735,1.666667,1.0,2.0,0.333333,0.57735,...,0.0,0.0,0.0,0.0,0.0,0.666667,0.0,1.0,0.333333,0.577350
3018,1.000000,1.0,1.0,0.000000,0.00000,1.666667,1.0,2.0,0.333333,0.57735,...,0.0,0.0,0.0,0.0,0.0,2.666667,1.0,4.0,2.333333,1.527525
3020,0.666667,0.0,1.0,0.333333,0.57735,1.666667,1.0,2.0,0.333333,0.57735,...,0.0,0.0,0.0,0.0,0.0,2.666667,2.0,4.0,1.333333,1.154701
3028,1.333333,1.0,2.0,0.333333,0.57735,2.000000,2.0,2.0,0.000000,0.00000,...,0.0,0.0,0.0,0.0,0.0,1.333333,0.0,3.0,2.333333,1.527525
3051,0.333333,0.0,1.0,0.333333,0.57735,1.000000,1.0,1.0,0.000000,0.00000,...,0.0,0.0,0.0,0.0,0.0,1.333333,1.0,2.0,0.333333,0.577350


In [ ]:
df_HY4_M = evaluate_models_10x10_oof_and_test(
    X_df=X_HY4_data_M, 
    y_df=y_HY4_data_M, 
    models=classification_models, 
    random_state=42
)

df_HY4_M.to_csv(full_set_path_HY4 / "HY4_M_StandardScaler.csv")

df_HY4_M_min_max = evaluate_models_10x10_oof_and_test(
    X_df=X_HY4_data_M, 
    y_df=y_HY4_data_M, 
    models=classification_models,
    scaler = MinMaxScaler(), 
    random_state=42
)
df_HY4_M_min_max.to_csv(full_set_path_HY4 / "HY4_M_MinMaxScaler.csv")


Evaluating decision_tree...
Evaluating random_forest...


In [ ]:
df_HY43_IM.head(10)

In [ ]:
df_HY43_IM_min_max.head(10)

## MCID


### FULL DOMAIN


In [ ]:
domain_cols = SC_cols + NM_cols + IM_cols + M_cols
X_MCID_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS3/FULL_FEATURES/X_MCID_full.csv", index_col=0)[domain_cols]
y_MCID_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS3/FULL_FEATURES/y_MCID_full.csv", index_col=0)
print("X_MCID_data shape:", X_MCID_data.shape)
print("y_MCID_data shape:", y_MCID_data.shape)
X_MCID_data.head()

In [ ]:
df_MCID_full = evaluate_models_10x10_oof_and_test(
    X_df=X_MCID_data, 
    y_df=y_MCID_data, 
    models=classification_models, 
    random_state=42
)

df_MCID_full.to_csv(full_set_path_MCID / "MCID_full_StandardScaler.csv")

df_MCID_full_min_max = evaluate_models_10x10_oof_and_test(
    X_df=X_MCID_data, 
    y_df=y_MCID_data, 
    models=classification_models,
    scaler = MinMaxScaler(), 
    random_state=42
)
df_MCID_full_min_max.to_csv(full_set_path_MCID / "MCID_full_MinMaxScaler.csv")

In [ ]:
df_MCID_full.head(10)

In [ ]:
df_MCID_full_min_max.head(10)

### NON MOTOR DOMAIN


In [ ]:
domain_cols = NM_cols 
X_MCID_data_NM = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS3/FULL_FEATURES/X_MCID_full.csv", index_col=0)[domain_cols]
y_MCID_data_NM = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS3/FULL_FEATURES/y_MCID_full.csv", index_col=0)
print("X_MCID_data_NM shape:", X_MCID_data_NM.shape)
print("y_MCID_data_NM shape:", y_MCID_data_NM.shape)
X_MCID_data_NM.head()

In [ ]:
df_MCID_NM = evaluate_models_10x10_oof_and_test(
    X_df=X_MCID_data_NM, 
    y_df=y_MCID_data_NM, 
    models=classification_models, 
    random_state=42
)

df_MCID_NM.to_csv(full_set_path_MCID / "MCID_NM_StandardScaler.csv")

df_MCID_NM_min_max = evaluate_models_10x10_oof_and_test(
    X_df=X_MCID_data_NM, 
    y_df=y_MCID_data_NM, 
    models=classification_models,
    scaler = MinMaxScaler(), 
    random_state=42
)
df_MCID_NM_min_max.to_csv(full_set_path_MCID / "MCID_NM_MinMaxScaler.csv")

In [ ]:
df_MCID_NM.head(10)

In [ ]:
df_MCID_full_min_max.head(10)

### IMPACT MOTOR DOMAIN


In [ ]:
domain_cols = IM_cols 
X_MCID_data_IM = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS3/FULL_FEATURES/X_MCID_full.csv", index_col=0)[domain_cols]
y_MCID_data_IM = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS3/FULL_FEATURES/y_MCID_full.csv", index_col=0)
print("X_MCID_data_IM shape:", X_MCID_data_IM.shape)
print("y_MCID_data_IM shape:", y_MCID_data_IM.shape)
X_MCID_data_IM.head()

In [ ]:
df_MCID_IM = evaluate_models_10x10_oof_and_test(
    X_df=X_MCID_data_IM, 
    y_df=y_MCID_data_IM, 
    models=classification_models, 
    random_state=42
)

df_MCID_IM.to_csv(full_set_path_MCID / "MCID_IM_StandardScaler.csv")

df_MCID_IM_min_max = evaluate_models_10x10_oof_and_test(
    X_df=X_MCID_data_IM, 
    y_df=y_MCID_data_IM, 
    models=classification_models,
    scaler = MinMaxScaler(), 
    random_state=42
)
df_MCID_IM_min_max.to_csv(full_set_path_MCID / "MCID_IM_MinMaxScaler.csv")

In [ ]:
df_MCID_IM.head(10)

In [ ]:
df_MCID_IM_min_max.head(10)

### MOTOR DOMAIN EXAM


In [ ]:
domain_cols = M_cols 
X_MCID_data_M = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS3/FULL_FEATURES/X_MCID_full.csv", index_col=0)[domain_cols]
y_MCID_data_M = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS3/FULL_FEATURES/y_MCID_full.csv", index_col=0)
print("X_MCID_data_M shape:", X_MCID_data_M.shape)
print("y_MCID_data_M shape:", y_MCID_data_M.shape)
X_MCID_data_M.head()

In [ ]:
df_MCID_M = evaluate_models_10x10_oof_and_test(
    X_df=X_MCID_data_M, 
    y_df=y_MCID_data_M, 
    models=classification_models, 
    random_state=42
)

df_MCID_M.to_csv(full_set_path_MCID / "MCID_M_StandardScaler.csv")

df_MCID_M_min_max = evaluate_models_10x10_oof_and_test(
    X_df=X_MCID_data_M, 
    y_df=y_MCID_data_M, 
    models=classification_models,
    scaler = MinMaxScaler(), 
    random_state=42
)
df_MCID_M_min_max.to_csv(full_set_path_MCID / "MCID_M_MinMaxScaler.csv")

In [ ]:
df_MCID_M.head(10)

In [ ]:
df_MCID_M_min_max.head(10)

---

# USO DE OVERSAMPLING DUE TO THE UNBALNACED DATASET

In [ ]:
import matplotlib.pyplot as plt

y_MCID = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS3/FULL_FEATURES/y_MCID_full.csv", index_col=0)
y_HY3 = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY3_full.csv", index_col=0)
y_HY4 = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY4_full.csv", index_col=0)
y_HY43 = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY4_3_full.csv", index_col=0)
print("y_MCID shape:", y_MCID.shape)
print("y_HY3 shape:", y_HY3.shape)
print("y_HY4 shape:", y_HY4.shape)
print("y_HY4_3 shape:", y_HY43.shape)

datasets = {
    "MCID": y_MCID,
    "HY3": y_HY3,
    "HY4": y_HY4,
    'HY4_3': y_HY43
}

fig, axes = plt.subplots(1, 4, figsize=(20,5))

for ax, (name, df) in zip(axes, datasets.items()):
    
    counts = df.iloc[:,0].value_counts().sort_index()
    percentages = counts / counts.sum() * 100
    
    bars = ax.bar(counts.index.astype(str), counts.values)
    
    # añadir texto con conteo y porcentaje
    for i, (count, pct) in enumerate(zip(counts.values, percentages.values)):
        ax.text(i, count, f"{count} ({pct:.1f}%)", 
                ha='center', va='bottom')
    
    ax.set_title(name)
    ax.set_xlabel("Class")
    ax.set_ylabel("Count")

plt.suptitle("Class Distribution")
plt.tight_layout()
plt.show()

---


# AGNOSTIC FEATURE SELECTION

In [ ]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import mutual_info_classif, chi2
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)


# =========================================================
# Feature selectors
# =========================================================

class SpearmanSULOVSelector(BaseEstimator, TransformerMixin):
    """
    Elimina variables altamente correlacionadas usando correlación de Spearman.
    Entre dos variables correlacionadas, conserva la que tenga mayor mutual information con y.
    """
    def __init__(self, threshold=0.9, random_state=42):
        self.threshold = threshold
        self.random_state = random_state
        self.feature_names_in_ = None
        self.mi_scores_ = None
        self.vars_to_drop_ = None
        self.selected_features_ = None
        self.n_features_selected_ = None

    def fit(self, X, y):
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)

        self.feature_names_in_ = X.columns.to_list()

        mi = mutual_info_classif(X, y, random_state=self.random_state)
        self.mi_scores_ = pd.Series(mi, index=X.columns)

        corr_df = X.corr(method="spearman").abs()
        upper = corr_df.where(np.triu(np.ones(corr_df.shape), k=1).astype(bool))

        vars_to_drop = set()

        for col in upper.columns:
            correlated_features = upper.index[upper[col] > self.threshold].tolist()

            for row_feature in correlated_features:
                if row_feature in vars_to_drop or col in vars_to_drop:
                    continue

                if self.mi_scores_[row_feature] <= self.mi_scores_[col]:
                    vars_to_drop.add(row_feature)
                else:
                    vars_to_drop.add(col)

        self.vars_to_drop_ = list(vars_to_drop)
        self.selected_features_ = [c for c in X.columns if c not in self.vars_to_drop_]
        self.n_features_selected_ = len(self.selected_features_)

        return self

    def transform(self, X):
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X, columns=self.feature_names_in_)
        return X.drop(columns=self.vars_to_drop_, errors="ignore")


class SpearmanCorrelationDiscard(BaseEstimator, TransformerMixin):
    """
    Elimina variables altamente correlacionadas según Spearman,
    conservando la que tenga mayor correlación absoluta con el target.
    """
    def __init__(self, threshold=0.9):
        self.threshold = threshold
        self.vars_to_drop_ = None
        self.feature_names_in_ = None

    def fit(self, X, y=None):
        if y is None:
            raise ValueError("SpearmanCorrelationDiscard requiere y en fit().")

        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)

        self.feature_names_in_ = X.columns.to_list()

        data = X.copy()
        data["_target_"] = y

        corr_df = data.corr(method="spearman")

        mask = np.tril(np.ones(corr_df.shape), k=-1).astype(bool)

        corr_long = (
            corr_df.where(mask)
            .stack()
            .reset_index()
            .rename(columns={"level_0": "V1", "level_1": "V2", 0: "CORR"})
        )

        corr_long = corr_long[
            (corr_long["V1"] != "_target_") & (corr_long["V2"] != "_target_")
        ].copy()

        target_corr = corr_df["_target_"]

        corr_long["V1target"] = corr_long["V1"].map(target_corr)
        corr_long["V2target"] = corr_long["V2"].map(target_corr)

        corr_long["WORST_VAR"] = np.where(
            abs(corr_long["V1target"]) <= abs(corr_long["V2target"]),
            corr_long["V1"],
            corr_long["V2"]
        )

        discard_corr_long = corr_long.loc[corr_long["CORR"].abs() > self.threshold]
        self.vars_to_drop_ = list(set(discard_corr_long["WORST_VAR"]))

        return self

    def transform(self, X):
        if isinstance(X, pd.DataFrame):
            return X.drop(columns=self.vars_to_drop_, errors="ignore")

        X_df = pd.DataFrame(X, columns=self.feature_names_in_)
        X_df = X_df.drop(columns=self.vars_to_drop_, errors="ignore")
        return X_df


class MIThresholdSelector(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.01, random_state=42):
        self.threshold = threshold
        self.random_state = random_state
        self.support_ = None
        self.mi_scores_ = None
        self.feature_names_in_ = None

    def fit(self, X, y):
        if isinstance(X, pd.DataFrame):
            self.feature_names_in_ = X.columns.to_list()
            X_fit = X.values
        else:
            self.feature_names_in_ = None
            X_fit = X

        self.mi_scores_ = mutual_info_classif(X_fit, y, random_state=self.random_state)
        self.support_ = self.mi_scores_ >= self.threshold

        if not np.any(self.support_):
            self.support_[np.argmax(self.mi_scores_)] = True

        return self

    def transform(self, X):
        if isinstance(X, pd.DataFrame):
            cols = X.columns[np.asarray(self.support_)]
            return X.loc[:, cols]

        return X[:, self.support_]


class Chi2ThresholdSelector(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.0):
        self.threshold = threshold
        self.support_ = None
        self.chi2_scores_ = None
        self.pvalues_ = None
        self.feature_names_in_ = None

    def fit(self, X, y):
        if isinstance(X, pd.DataFrame):
            self.feature_names_in_ = X.columns.to_list()
            X_fit = X.values
        else:
            self.feature_names_in_ = None
            X_fit = X

        self.chi2_scores_, self.pvalues_ = chi2(X_fit, y)
        self.support_ = self.chi2_scores_ >= self.threshold

        if not np.any(self.support_):
            self.support_[np.argmax(self.chi2_scores_)] = True

        return self

    def transform(self, X):
        if isinstance(X, pd.DataFrame):
            cols = X.columns[np.asarray(self.support_)]
            return X.loc[:, cols]

        return X[:, self.support_]


class Chi2MIUnionTopKSelector(BaseEstimator, TransformerMixin):
    """
    Selecciona la unión entre:
    - top_k variables según chi2
    - top_k variables según mutual information

    Nota:
    - chi2 requiere valores no negativos, por eso este selector debe usarse
      después de un MinMaxScaler.
    """
    def __init__(self, top_k=100, random_state=42):
        self.top_k = top_k
        self.random_state = random_state
        self.feature_names_in_ = None
        self.chi2_scores_ = None
        self.mi_scores_ = None
        self.selected_features_ = None
        self.support_ = None
        self.n_features_selected_ = None

    def fit(self, X, y):
        if isinstance(X, pd.DataFrame):
            self.feature_names_in_ = X.columns.to_list()
            X_fit = X.values
        else:
            self.feature_names_in_ = None
            X_fit = X

        n_features = X_fit.shape[1]
        k = min(self.top_k, n_features)

        self.chi2_scores_, _ = chi2(X_fit, y)
        self.mi_scores_ = mutual_info_classif(X_fit, y, random_state=self.random_state)

        top_chi2_idx = np.argsort(self.chi2_scores_)[::-1][:k]
        top_mi_idx = np.argsort(self.mi_scores_)[::-1][:k]

        selected_idx = np.union1d(top_chi2_idx, top_mi_idx)

        self.support_ = np.zeros(n_features, dtype=bool)
        self.support_[selected_idx] = True
        self.n_features_selected_ = int(self.support_.sum())

        if self.feature_names_in_ is not None:
            self.selected_features_ = [self.feature_names_in_[i] for i in selected_idx]
        else:
            self.selected_features_ = selected_idx.tolist()

        return self

    def transform(self, X):
        if isinstance(X, pd.DataFrame):
            cols = X.columns[np.asarray(self.support_)]
            return X.loc[:, cols]

        return X[:, self.support_]


# =========================================================
# Helpers
# =========================================================

def build_pipeline(
    estimator,
    selector_name,
    spearman_threshold=0.9,
    mi_threshold=0.01,
    chi2_threshold=3.84,
    chi2_mi_top_k=100,
    random_state=42,
):
    if selector_name == "spearman_corr":
        return Pipeline([
            ("selector", SpearmanCorrelationDiscard(threshold=spearman_threshold)),
            ("scaler", MinMaxScaler()),
            ("model", clone(estimator)),
        ])

    elif selector_name == "spearman_sulov":
        return Pipeline([
            ("selector", SpearmanSULOVSelector(
                threshold=spearman_threshold,
                random_state=random_state
            )),
            ("scaler", MinMaxScaler()),
            ("model", clone(estimator)),
        ])

    elif selector_name == "mutual_info":
        return Pipeline([
            ("selector", MIThresholdSelector(
                threshold=mi_threshold,
                random_state=random_state
            )),
            ("scaler", MinMaxScaler()),
            ("model", clone(estimator)),
        ])

    elif selector_name == "chi2":
        return Pipeline([
            ("minmax_before_chi2", MinMaxScaler()),
            ("selector", Chi2ThresholdSelector(threshold=chi2_threshold)),
            ("scaler", MinMaxScaler()),
            ("model", clone(estimator)),
        ])

    elif selector_name == "chi2_mi_union_topk":
        return Pipeline([
            ("minmax_before_union", MinMaxScaler()),
            ("selector", Chi2MIUnionTopKSelector(
                top_k=chi2_mi_top_k,
                random_state=random_state
            )),
            ("scaler", MinMaxScaler()),
            ("model", clone(estimator)),
        ])

    else:
        raise ValueError(f"Selector no soportado: {selector_name}")


def safe_multiclass_auc(y_true, y_proba, average="macro"):
    try:
        present_classes = np.unique(y_true)
        if len(present_classes) < 2:
            return np.nan

        y_proba_used = y_proba[:, present_classes]
        mapper = {cls: i for i, cls in enumerate(present_classes)}
        y_true_mapped = np.array([mapper[v] for v in y_true])

        return roc_auc_score(
            y_true_mapped,
            y_proba_used,
            multi_class="ovr",
            average=average
        )
    except Exception:
        return np.nan


def compute_metrics(y_true, y_pred, y_proba):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "F1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "AUC_macro": safe_multiclass_auc(y_true, y_proba, average="macro"),
        "Precision_weighted": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "Recall_weighted": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "F1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "AUC_weighted": safe_multiclass_auc(y_true, y_proba, average="weighted"),
    }


def summarize(metrics_list, suffix="CV", decimals=4):
    df = pd.DataFrame(metrics_list)
    mean = df.mean(numeric_only=True)
    std = df.std(ddof=1, numeric_only=True)

    return {
        f"{c}_{suffix}": f"{mean[c]:.{decimals}f} ± {std[c]:.{decimals}f}"
        for c in mean.index
    }


def _predict_proba_aligned(pipe, X, classes_global):
    proba = pipe.predict_proba(X)
    model_classes = pipe.named_steps["model"].classes_

    aligned = np.zeros((len(X), len(classes_global)), dtype=float)
    class_to_global_idx = {c: i for i, c in enumerate(classes_global)}

    for local_idx, cls in enumerate(model_classes):
        aligned[:, class_to_global_idx[cls]] = proba[:, local_idx]

    return aligned


# =========================================================
# Función principal
# =========================================================

def evaluate_models_oof_and_test_with_fs(
    X_df: pd.DataFrame,
    y_df: pd.DataFrame,
    models: dict,
    outer_splits: int = 10,
    inner_splits: int = 10,
    test_size: float = 0.3,
    random_state: int = 42,
    decimals: int = 4,
    selectors=None,
    spearman_threshold: float = 0.9,
    mi_threshold: float = 0.01,
    chi2_threshold: float = 3.84,
    chi2_mi_top_k: int = 100,
):
    if selectors is None:
        selectors = [
            "spearman_corr",
            "mutual_info",
            "spearman_sulov",
            "chi2",
            "chi2_mi_union_topk",
        ]

    X = X_df.copy()
    y = y_df.iloc[:, 0].to_numpy()

    models = {
        name: model
        for name, model in models.items()
        if hasattr(model, "predict_proba")
    }

    if len(models) == 0:
        raise ValueError("Ningún modelo tiene el método predict_proba().")

    classes_global = np.unique(y)
    outer = StratifiedShuffleSplit(
        n_splits=outer_splits,
        test_size=test_size,
        random_state=random_state
    )

    all_rows = []

    for model_name, estimator in models.items():
        for selector_name in selectors:
            print(f"Evaluating {model_name} + {selector_name}...")

            test_metrics_all = []
            cv_metrics_all = []

            for train_idx, test_idx in outer.split(X, y):
                X_train = X.iloc[train_idx].copy()
                X_test = X.iloc[test_idx].copy()
                y_train = y[train_idx]
                y_test = y[test_idx]

                _, class_counts = np.unique(y_train, return_counts=True)
                min_class_count = class_counts.min()
                effective_inner_splits = min(inner_splits, min_class_count)

                if effective_inner_splits < 2:
                    raise ValueError(
                        "No hay suficientes muestras por clase para realizar StratifiedKFold."
                    )

                inner = StratifiedKFold(
                    n_splits=effective_inner_splits,
                    shuffle=True,
                    random_state=random_state
                )

                oof_pred = np.empty(len(y_train), dtype=y_train.dtype)
                oof_proba = np.zeros((len(y_train), len(classes_global)), dtype=float)

                for tr_idx, val_idx in inner.split(X_train, y_train):
                    X_tr = X_train.iloc[tr_idx].copy()
                    X_val = X_train.iloc[val_idx].copy()
                    y_tr = y_train[tr_idx]

                    pipe = build_pipeline(
                        estimator=estimator,
                        selector_name=selector_name,
                        spearman_threshold=spearman_threshold,
                        mi_threshold=mi_threshold,
                        chi2_threshold=chi2_threshold,
                        chi2_mi_top_k=chi2_mi_top_k,
                        random_state=random_state,
                    )

                    pipe.fit(X_tr, y_tr)

                    oof_pred[val_idx] = pipe.predict(X_val)
                    oof_proba[val_idx] = _predict_proba_aligned(pipe, X_val, classes_global)

                cv_metrics_all.append(
                    compute_metrics(
                        y_true=y_train,
                        y_pred=oof_pred,
                        y_proba=oof_proba
                    )
                )

                pipe_full = build_pipeline(
                    estimator=estimator,
                    selector_name=selector_name,
                    spearman_threshold=spearman_threshold,
                    mi_threshold=mi_threshold,
                    chi2_threshold=chi2_threshold,
                    chi2_mi_top_k=chi2_mi_top_k,
                    random_state=random_state,
                )

                pipe_full.fit(X_train, y_train)

                test_pred = pipe_full.predict(X_test)
                test_proba = _predict_proba_aligned(pipe_full, X_test, classes_global)

                test_metrics_all.append(
                    compute_metrics(
                        y_true=y_test,
                        y_pred=test_pred,
                        y_proba=test_proba
                    )
                )

            row = {
                "F1_macro_CV": summarize(cv_metrics_all, suffix="CV", decimals=decimals)["F1_macro_CV"],
                "F1_macro_Testing": summarize(test_metrics_all, suffix="Testing", decimals=decimals)["F1_macro_Testing"],
                "Model": model_name,
                "Feature_Selection": selector_name,
            }

            row.update(summarize(test_metrics_all, suffix="Testing", decimals=decimals))
            row.update(summarize(cv_metrics_all, suffix="CV", decimals=decimals))

            all_rows.append(row)

    df_final_summary = pd.DataFrame(all_rows)[[
        "Model",
        "Feature_Selection",
        "F1_macro_CV",
        "F1_macro_Testing",
        "Accuracy_Testing",
        "Precision_macro_Testing",
        "Recall_macro_Testing",
        "AUC_macro_Testing",
        "Precision_weighted_Testing",
        "Recall_weighted_Testing",
        "F1_weighted_Testing",
        "AUC_weighted_Testing",
        "Accuracy_CV",
        "Precision_macro_CV",
        "Recall_macro_CV",
        "AUC_macro_CV",
        "Precision_weighted_CV",
        "Recall_weighted_CV",
        "F1_weighted_CV",
        "AUC_weighted_CV",
    ]]

    return df_final_summary



## HY3

In [ ]:
domain_cols = SC_cols + NM_cols + IM_cols + M_cols
X_HY3_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_HY3_full.csv", index_col=0)[domain_cols]
y_HY3_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY3_full.csv", index_col=0)
print("X_HY3_data shape:", X_HY3_data.shape)
print("y_HY3_data shape:", y_HY3_data.shape)
X_HY3_data.head()

In [ ]:
HY3_FS_results = evaluate_models_oof_and_test_with_fs(
    X_df=X_HY3_data,
    y_df=y_HY3_data,
    models=classification_models)
HY3_FS_results.to_csv(feature_selection_path_HY3 / "HY3_full_FS_results.csv", index=False)
HY3_FS_results.head(10)

## HY43

In [ ]:
domain_cols = SC_cols + NM_cols + IM_cols + M_cols
X_HY43_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_HY4_full.csv", index_col=0)[domain_cols]
y_HY43_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY4_3_full.csv", index_col=0)
print("X_HY43_data shape:", X_HY43_data.shape)
print("y_HY43_data shape:", y_HY43_data.shape)
X_HY43_data.head()

In [ ]:
HY43_FS_results = evaluate_models_oof_and_test_with_fs(
    X_df=X_HY43_data,
    y_df=y_HY43_data,
    models=classification_models)
HY43_FS_results.to_csv(feature_selection_path_HY4 / "HY43_full_FS_results.csv", index=False)
HY43_FS_results.head(10)

## HY4


In [ ]:
domain_cols = SC_cols + NM_cols + IM_cols + M_cols
X_HY4_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_HY4_full.csv", index_col=0)[domain_cols]
y_HY4_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY4_full.csv", index_col=0)
print("X_HY4_data shape:", X_HY4_data.shape)
print("y_HY4_data shape:", y_HY4_data.shape)
X_HY4_data.head()

In [ ]:
HY4_FS_results = evaluate_models_oof_and_test_with_fs(
    X_df=X_HY4_data,
    y_df=y_HY4_data,
    models=classification_models)
HY4_FS_results.to_csv(feature_selection_path_HY4 / "HY4_full_FS_results.csv", index=False)
HY4_FS_results.head(10)

## MCID

In [ ]:
domain_cols = SC_cols + NM_cols + IM_cols + M_cols
X_MCID_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS3/FULL_FEATURES/X_MCID_full.csv", index_col=0)[domain_cols]
y_MCID_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS3/FULL_FEATURES/y_MCID_full.csv", index_col=0)
print("X_MCID_data shape:", X_MCID_data.shape)
print("y_MCID_data shape:", y_MCID_data.shape)
X_MCID_data.head()

In [ ]:
MCID_FS_results = evaluate_models_oof_and_test_with_fs(
    X_df=X_MCID_data,
    y_df=y_MCID_data,
    models=classification_models)
MCID_FS_results.to_csv(feature_selection_path_MCID / "MCID_full_FS_results.csv", index=False)
MCID_FS_results.head(10)

---


# BAYESIAN OPTIMIZATION
- FEATURE SELECTION: chi2_mi_union_topk
- HYPERPARAMETER TURNING

In [ ]:
import numpy as np
import pandas as pd

from tqdm.auto import tqdm

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import mutual_info_classif, chi2
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

from skopt import BayesSearchCV
from skopt.space import Real, Integer, Categorical


class Chi2MIUnionTopKSelector(BaseEstimator, TransformerMixin):
    """
    Selecciona la unión entre:
    - top_k variables según chi2
    - top_k variables según mutual information

    Nota:
    - chi2 requiere valores no negativos, por eso este selector debe usarse
      después de un MinMaxScaler.
    """
    def __init__(self, top_k=100, random_state=42):
        self.top_k = top_k
        self.random_state = random_state
        self.feature_names_in_ = None
        self.chi2_scores_ = None
        self.mi_scores_ = None
        self.selected_features_ = None
        self.support_ = None
        self.n_features_selected_ = None

    def fit(self, X, y):
        if isinstance(X, pd.DataFrame):
            self.feature_names_in_ = X.columns.to_list()
            X_fit = X.values
        else:
            self.feature_names_in_ = None
            X_fit = X

        n_features = X_fit.shape[1]
        k = min(self.top_k, n_features)

        self.chi2_scores_, _ = chi2(X_fit, y)
        self.mi_scores_ = mutual_info_classif(X_fit, y, random_state=self.random_state)

        top_chi2_idx = np.argsort(self.chi2_scores_)[::-1][:k]
        top_mi_idx = np.argsort(self.mi_scores_)[::-1][:k]

        selected_idx = np.union1d(top_chi2_idx, top_mi_idx)

        self.support_ = np.zeros(n_features, dtype=bool)
        self.support_[selected_idx] = True
        self.n_features_selected_ = int(self.support_.sum())

        if self.feature_names_in_ is not None:
            self.selected_features_ = [self.feature_names_in_[i] for i in selected_idx]
        else:
            self.selected_features_ = selected_idx.tolist()

        return self

    def transform(self, X):
        if isinstance(X, pd.DataFrame):
            cols = X.columns[np.asarray(self.support_)]
            return X.loc[:, cols]
        return X[:, self.support_]


def evaluate_models_nested_bayes(
    X_df: pd.DataFrame,
    y_df: pd.DataFrame,
    models: dict,
    outer_splits: int = 5,
    inner_splits: int = 10,
    test_size: float = 0.3,
    random_state: int = 42,
    decimals: int = 4,
    n_iter_search: int = 40,
    n_jobs_search: int = -1,
    selector_name: str = None,
    chi2_mi_top_k: int = 100,
):
    y = y_df.iloc[:, 0].to_numpy()
    X = X_df.to_numpy()
    classes = np.unique(y)

    def build_pipeline(estimator, selector_name=None):
        if selector_name is None:
            return Pipeline([
                ("scaler", MinMaxScaler()),
                ("model", clone(estimator)),
            ])

        elif selector_name == "chi2_mi_union_topk":
            return Pipeline([
                ("minmax_before_union", MinMaxScaler()),
                ("selector", Chi2MIUnionTopKSelector(
                    top_k=chi2_mi_top_k,
                    random_state=random_state
                )),
                ("scaler", MinMaxScaler()),
                ("model", clone(estimator)),
            ])

        else:
            raise ValueError(f"Selector no soportado: {selector_name}")

    def get_search_spaces(model_name):
        spaces = {
            "decision_tree": {
                "model__max_depth": Integer(2, 30),
                "model__min_samples_split": Integer(2, 20),
                "model__min_samples_leaf": Integer(1, 10),
                "model__criterion": Categorical(["gini", "entropy"]),
                "model__max_features": Categorical([None, "sqrt"]),
                "model__class_weight": Categorical([None, "balanced"]),
            },

            "random_forest": {
                "model__n_estimators": Integer(200, 800),
                "model__max_depth": Integer(4, 30),
                "model__min_samples_split": Integer(2, 20),
                "model__min_samples_leaf": Integer(1, 10),
                "model__max_features": Categorical(["sqrt", "log2"]),
                "model__bootstrap": Categorical([True]),
                "model__class_weight": Categorical([None, "balanced", "balanced_subsample"]),
            },

            "extra_trees": {
                "model__n_estimators": Integer(200, 800),
                "model__max_depth": Integer(4, 30),
                "model__min_samples_split": Integer(2, 20),
                "model__min_samples_leaf": Integer(1, 10),
                "model__max_features": Categorical(["sqrt", "log2"]),
                "model__class_weight": Categorical([None, "balanced", "balanced_subsample"]),
            },

            "xgboost": {
                "model__n_estimators": Integer(200, 600),
                "model__max_depth": Integer(3, 10),
                "model__learning_rate": Real(0.01, 0.3, prior="log-uniform"),
                "model__subsample": Real(0.6, 1.0),
                "model__colsample_bytree": Real(0.6, 1.0),
                "model__min_child_weight": Integer(1, 10),
                "model__gamma": Real(1e-8, 5.0, prior="log-uniform"),
                "model__reg_alpha": Real(1e-8, 5.0, prior="log-uniform"),
                "model__reg_lambda": Real(1e-8, 5.0, prior="log-uniform"),
            },

            "adaboost": {
                "model__n_estimators": Integer(50, 500),
                "model__learning_rate": Real(0.01, 1.0, prior="log-uniform"),
            },

            "svm": {
                "model__C": Real(1e-3, 1e3, prior="log-uniform"),
                "model__gamma": Real(1e-5, 1.0, prior="log-uniform"),
                "model__kernel": Categorical(["rbf"]),
                "model__class_weight": Categorical([None, "balanced"]),
            },

            "logistic_regression": {
                "model__C": Real(1e-4, 1e2, prior="log-uniform"),
                "model__solver": Categorical(["lbfgs", "saga"]),
                "model__penalty": Categorical(["l2"]),
                "model__class_weight": Categorical([None, "balanced"]),
            },

            "knn": {
                "model__n_neighbors": Integer(3, 51),
                "model__weights": Categorical(["uniform", "distance"]),
                "model__p": Integer(1, 2),
            },

            "gaussian_nb": {
                "model__var_smoothing": Real(1e-10, 1e-6, prior="log-uniform"),
            },
        }

        if model_name not in spaces:
            raise ValueError(f"No search space definido para {model_name}")

        return spaces[model_name]

    def compute_metrics(y_true, y_pred, y_proba):
        return {
            "Accuracy": accuracy_score(y_true, y_pred),
            "Precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
            "Recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
            "F1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
            "AUC_macro": roc_auc_score(y_true, y_proba, multi_class="ovr", average="macro"),
            "Precision_weighted": precision_score(y_true, y_pred, average="weighted", zero_division=0),
            "Recall_weighted": recall_score(y_true, y_pred, average="weighted", zero_division=0),
            "F1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
            "AUC_weighted": roc_auc_score(y_true, y_proba, multi_class="ovr", average="weighted"),
        }

    def summarize(metrics_list, suffix):
        df = pd.DataFrame(metrics_list)
        mean = df.mean(numeric_only=True)
        std = df.std(ddof=1, numeric_only=True)
        return {
            f"{col}_{suffix}": f"{mean[col]:.{decimals}f} ± {std[col]:.{decimals}f}"
            for col in df.columns
        }

    class TqdmBayesCallback:
        """
        Callback para actualizar una barra tqdm en cada iteración
        del BayesSearchCV.
        """
        def __init__(self, pbar):
            self.pbar = pbar

        def __call__(self, res):
            best_cv = -np.min(res.func_vals)
            self.pbar.update(1)
            self.pbar.set_postfix({
                "best_inner_cv_f1": f"{best_cv:.4f}"
            })
            return False

    outer = StratifiedShuffleSplit(
        n_splits=outer_splits,
        test_size=test_size,
        random_state=random_state
    )

    test_summary_rows = []
    cv_summary_rows = []
    best_params_rows = []

    for model_idx, (model_name, estimator) in enumerate(models.items(), start=1):
        print(f"\nEvaluating {model_name} with Bayesian Search...")

        test_metrics_all = []
        cv_metrics_all = []
        best_params_per_outer_fold = []

        search_spaces = get_search_spaces(model_name)

        outer_pbar = tqdm(
            total=outer_splits,
            desc=f"[{model_idx}/{len(models)}] {model_name} OUTER",
            position=0,
            leave=True
        )

        for fold_id, (train_idx, test_idx) in enumerate(outer.split(X, y), start=1):
            X_train, y_train = X[train_idx], y[train_idx]
            X_test, y_test = X[test_idx], y[test_idx]

            inner = StratifiedKFold(
                n_splits=inner_splits,
                shuffle=True,
                random_state=random_state
            )

            base_pipeline = build_pipeline(estimator, selector_name=selector_name)

            inner_pbar = tqdm(
                total=n_iter_search,
                desc=f"Outer fold {fold_id}/{outer_splits} | Bayes {n_iter_search} iters | inner_cv={inner_splits}",
                position=1,
                leave=False
            )

            opt = BayesSearchCV(
                estimator=base_pipeline,
                search_spaces=search_spaces,
                n_iter=n_iter_search,
                scoring="f1_macro",
                cv=inner,
                n_jobs=n_jobs_search,
                refit=True,
                random_state=random_state,
                verbose=0,
            )

            callback = TqdmBayesCallback(inner_pbar)
            opt.fit(X_train, y_train, callback=callback)

            inner_pbar.close()

            best_model = opt.best_estimator_
            best_params_per_outer_fold.append(opt.best_params_)

            # Guardar info del selector si existe
            if selector_name == "chi2_mi_union_topk":
                selector = best_model.named_steps["selector"]
                best_params_per_outer_fold[-1]["selected_features_count"] = selector.n_features_selected_
                best_params_per_outer_fold[-1]["selected_features"] = selector.selected_features_

            cv_metrics_all.append({
                "F1_macro": opt.best_score_,
            })

            test_pred = best_model.predict(X_test)
            test_proba_raw = best_model.predict_proba(X_test)

            test_classes = best_model.named_steps["model"].classes_
            test_proba = np.zeros((len(y_test), len(classes)))

            for j, cls in enumerate(test_classes):
                test_proba[:, np.where(classes == cls)[0][0]] = test_proba_raw[:, j]

            outer_metrics = compute_metrics(y_test, test_pred, test_proba)
            test_metrics_all.append(outer_metrics)

            avg_outer_f1 = np.mean([m["F1_macro"] for m in test_metrics_all])
            avg_inner_cv_f1 = np.mean([m["F1_macro"] for m in cv_metrics_all])

            outer_pbar.update(1)
            outer_pbar.set_postfix({
                "fold": f"{fold_id}/{outer_splits}",
                "avg_outer_f1": f"{avg_outer_f1:.4f}",
                "avg_inner_cv_f1": f"{avg_inner_cv_f1:.4f}",
            })

        outer_pbar.close()

        test_summary_rows.append(
            pd.Series(summarize(test_metrics_all, "Testing"), name=model_name)
        )

        cv_summary_rows.append(
            pd.Series(summarize(cv_metrics_all, "CV"), name=model_name)
        )

        best_params_rows.append(
            pd.Series({"Best_Params_Outer_Folds": best_params_per_outer_fold}, name=model_name)
        )

    df_test_summary = pd.DataFrame(test_summary_rows)[[
        "Accuracy_Testing",
        "Precision_macro_Testing",
        "Recall_macro_Testing",
        "F1_macro_Testing",
        "AUC_macro_Testing",
        "Precision_weighted_Testing",
        "Recall_weighted_Testing",
        "F1_weighted_Testing",
        "AUC_weighted_Testing"
    ]]

    df_cv_summary = pd.DataFrame(cv_summary_rows)[[
        "F1_macro_CV"
    ]]

    df_best_params = pd.DataFrame(best_params_rows)

    df_final_summary = pd.concat(
        [df_test_summary, df_cv_summary, df_best_params],
        axis=1
    )

    return df_final_summary

## HY3

In [ ]:
domain_cols = SC_cols + NM_cols + IM_cols + M_cols
X_HY3_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_HY3_full.csv", index_col=0)[domain_cols]
y_HY3_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY3_full.csv", index_col=0)
print("X_HY3_data shape:", X_HY3_data.shape)
print("y_HY3_data shape:", y_HY3_data.shape)
X_HY3_data.head()

In [ ]:
HY3_results = evaluate_models_nested_bayes(
    X_df=X_HY3_data,
    y_df=y_HY3_data,
    models=classification_models,
    selector_name="chi2_mi_union_topk",   
    chi2_mi_top_k=100,                    
    outer_splits=10,
    inner_splits=10,
    n_iter_search=40,
)
HY3_results.to_csv(feature_selection_path_HY3 / "HY3_nested_bayes_chi2_mi_union_topk_results.csv", index=True)
HY3_results.head(10)

In [ ]:
HY3_results = evaluate_models_nested_bayes(
    X_df=X_HY3_data,
    y_df=y_HY3_data,
    models=classification_models,
    selector_name=None,   
    chi2_mi_top_k=100,                    
    outer_splits=10,
    inner_splits=10,
    n_iter_search=40,
)
HY3_results.to_csv(feature_selection_path_HY3 / "HY3_nested_bayes_NOFS_results.csv", index=True)
HY3_results.head(10)

## HY43

In [ ]:
domain_cols = SC_cols + NM_cols + IM_cols + M_cols
X_HY43_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_HY4_full.csv", index_col=0)[domain_cols]
y_HY43_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY4_3_full.csv", index_col=0)
print("X_HY43_data shape:", X_HY43_data.shape)
print("y_HY43_data shape:", y_HY43_data.shape)
X_HY43_data.head()

In [ ]:
HY43_results = evaluate_models_nested_bayes(
    X_df=X_HY43_data,
    y_df=y_HY43_data,
    models=classification_models,
    selector_name="chi2_mi_union_topk",   
    chi2_mi_top_k=100,                    
    outer_splits=10,
    inner_splits=10,
    n_iter_search=40,
)
HY43_results.to_csv(feature_selection_path_HY4 / "HY43_nested_bayes_chi2_mi_union_topk_results.csv", index=True)
HY43_results.head(10)

In [ ]:
HY43_results = evaluate_models_nested_bayes(
    X_df=X_HY43_data,
    y_df=y_HY43_data,
    models=classification_models,
    selector_name=None,   
    chi2_mi_top_k=100,                    
    outer_splits=10,
    inner_splits=10,
    n_iter_search=40,
)
HY43_results.to_csv(feature_selection_path_HY4 / "HY43_nested_bayes_NOFS_results.csv", index=True)
HY43_results.head(10)

## HY4

In [ ]:
domain_cols = SC_cols + NM_cols + IM_cols + M_cols
X_HY4_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_HY4_full.csv", index_col=0)[domain_cols]
y_HY4_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY4_full.csv", index_col=0)
print("X_HY4_data shape:", X_HY4_data.shape)
print("y_HY4_data shape:", y_HY4_data.shape)
X_HY4_data.head()

In [ ]:
HY4_results = evaluate_models_nested_bayes(
    X_df=X_HY4_data,
    y_df=y_HY4_data,
    models=classification_models,
    selector_name="chi2_mi_union_topk",   
    chi2_mi_top_k=100,                    
    outer_splits=10,
    inner_splits=10,
    n_iter_search=40,
)
HY4_results.to_csv(feature_selection_path_HY4 / "HY4_nested_bayes_chi2_mi_union_topk_results.csv", index=True)
HY4_results.head(10)

In [ ]:
HY4_results = evaluate_models_nested_bayes(
    X_df=X_HY4_data,
    y_df=y_HY4_data,
    models=classification_models,
    selector_name=None,   
    chi2_mi_top_k=100,                    
    outer_splits=10,
    inner_splits=10,
    n_iter_search=40,
)
HY4_results.to_csv(feature_selection_path_HY4 / "HY4_nested_bayes_NOFS_results.csv", index=True)
HY4_results.head(10)

## MCID

In [ ]:
domain_cols = SC_cols + NM_cols + IM_cols + M_cols
X_MCID_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS3/FULL_FEATURES/X_MCID_full.csv", index_col=0)[domain_cols]
y_MCID_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS3/FULL_FEATURES/y_MCID_full.csv", index_col=0)
print("X_MCID_data shape:", X_MCID_data.shape)
print("y_MCID_data shape:", y_MCID_data.shape)
X_MCID_data.head()

In [ ]:
MCID_results = evaluate_models_nested_bayes(
    X_df=X_MCID_data,
    y_df=y_MCID_data,
    models=classification_models,
    selector_name="chi2_mi_union_topk",   
    chi2_mi_top_k=100,                    
    outer_splits=10,
    inner_splits=10,
    n_iter_search=40,
)
MCID_results.to_csv(feature_selection_path_MCID / "MCID_nested_bayes_chi2_mi_union_topk_results.csv", index=True)
MCID_results.head(10)

In [ ]:
MCID_results = evaluate_models_nested_bayes(
    X_df=X_MCID_data,
    y_df=y_MCID_data,
    models=classification_models,
    selector_name=None,   
    chi2_mi_top_k=100,                    
    outer_splits=10,
    inner_splits=10,
    n_iter_search=40,
)
MCID_results.to_csv(feature_selection_path_MCID / "MCID_nested_bayes_NOFS_results.csv", index=True)
MCID_results.head(10)